# Intuit QuickBooks Upgrade Targeting

**Tools:** Python · Polars · pyrsm · scikit-learn · XGBoost · Plotnine  
**Models:** Logistic Regression · Logistic Regression with Interactions · Neural Network (MLP) · Random Forest · XGBoost  
**Techniques:** Response Modeling · Cross-Validation · Profit Optimization · Mailing Policy Design · ROME Analysis

Solo-led MSBA team project focused on predicting response to a QuickBooks upgrade mailing and translating model scores into a profitable wave-2 targeting policy.

## At a Glance

- Built and compared five mailing-response models on the Intuit campaign dataset
- Tuned XGBoost delivered the strongest holdout discrimination with AUC around **0.788**
- Evaluated break-even and half-threshold mailing policies on the holdout set
- Best deployment rule: **`mail_xgb_50thr`**, with estimated profit around **449.5K** and **ROME 1.50**

## Setup

This notebook expects the project dependencies listed in `requirements.txt` and reads the portfolio data files from `data/`. Run the notebook top to bottom to reproduce the analysis.

## Author Note

This public version adapts the original academic case submission into a portfolio case study. The goal is to show the modeling workflow, deployment logic, and campaign economics clearly for a recruiter or hiring-manager audience.


In [1]:
import polars as pl
import polars.selectors as cs
import pyrsm as rsm
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold
from plotnine import ggplot, aes, geom_line, geom_hline, labs, theme_bw
from plotnine import geom_col, geom_text, theme_minimal, coord_flip


In [2]:
## Load the data
intuit75k = pl.read_parquet("data/intuit75k.parquet")
intuit75k.head()

id,zip5,zip_bins,sex,bizflag,numords,dollars,last,sincepurch,version1,owntaxprod,upgraded,res1,training,res1_yes
i32,str,i32,cat,i32,i32,f64,i32,i32,i32,i32,i32,cat,i32,i64
1,"""94553""",18,"""Male""",0,2,109.5,5,12,0,0,0,"""No""",1,0
2,"""53190""",10,"""Unknown""",0,1,69.5,4,3,0,0,0,"""No""",0,0
3,"""37091""",8,"""Male""",0,4,93.0,14,29,0,0,1,"""No""",0,0
4,"""02125""",1,"""Male""",0,1,22.0,17,1,0,0,0,"""No""",1,0
5,"""60201""",11,"""Male""",0,1,24.5,2,3,0,0,0,"""No""",0,0


In [3]:
rsm.md("data/intuit75k_description.md")

## Intuit: Quickbooks upgrade

The purpose of this exercise is to gain experience modeling the response to an upsell campaign. The `intuit75k.parquet` file contains data on 75,000 (small) businesses that were selected randomly from the 801,821 that were sent the wave-1 mailing. The mailing contained an offer to upgrade to the latest version of the Quickbooks software.

Variable `res1` denotes which of these businesses responded to the mailing by purchasing Quickbooks version 3.0 from "Intuit Direct". Note that Intuit Direct sells products directly to its customers rather than through a retailer. Use the available data to predict which businesses that did not respond to the wave-1 mailing, are most likely to respond to the wave-2 mailing. Note that variables were added, deleted, and recoded so please ignore the variable descriptions in Exhibit 3 in the case pdf. Instead, use the variable descriptions below:

## Variable description

* id: Small business customer ID
* zip5: 5-Digit ZIP Code (00000=unknown, 99999=international ZIPs).
* zip_bins: Zip-code bins (20 approx. equal sized bins from lowest to highest zip code number)
* sex: Gender Identity "Female", "Male", or "Unknown"
* bizflag: Business Flag. Address contains a Business name (1 = yes, 0 = no or unknown).
* numords: Number of orders from Intuit Direct in the previous 36 months
* dollars: Total $ ordered from Intuit Direct in the previous 36 months
* last: Time (in months) since last order from Intuit Direct in previous 36 months
* sincepurch: Time (in months) since original (not upgrade) purchase of Quickbooks
* version1: Is 1 if customer's current Quickbooks is version 1, 0 if version 2
* owntaxprod: Is 1 if customer purchased tax software, 0 otherwise
* upgraded: Is 1 if customer upgraded from Quickbooks vs. 1 to vs. 2
* res1: Response to wave 1 mailing ("Yes" if responded else "No")
* training: 70/30 split, 1 for training sample, 0 for test sample

In [4]:
intuit75k.head()

id,zip5,zip_bins,sex,bizflag,numords,dollars,last,sincepurch,version1,owntaxprod,upgraded,res1,training,res1_yes
i32,str,i32,cat,i32,i32,f64,i32,i32,i32,i32,i32,cat,i32,i64
1,"""94553""",18,"""Male""",0,2,109.5,5,12,0,0,0,"""No""",1,0
2,"""53190""",10,"""Unknown""",0,1,69.5,4,3,0,0,0,"""No""",0,0
3,"""37091""",8,"""Male""",0,4,93.0,14,29,0,0,1,"""No""",0,0
4,"""02125""",1,"""Male""",0,1,22.0,17,1,0,0,0,"""No""",1,0
5,"""60201""",11,"""Male""",0,1,24.5,2,3,0,0,0,"""No""",0,0


### **Visualize Data**

In [5]:
ds = rsm.eda.distr(intuit75k.drop("id", "zip5"))
ds.plot()

## Distribution Interpretation

The raw feature distributions suggest a classic direct-marketing response problem: response is likely rare, customer value is skewed, and several behavioral variables such as prior orders, recency, and spending are likely to matter more than simple demographics. That means the business value of modeling will come from **ranking customers well**, not from trying to explain every individual conversion.


### **Associations**

Zip_bins and resp1

In [6]:
ct = rsm.basics.cross_tabs(
    {"intuit75k": intuit75k.filter(pl.col("training") == 1)}, "zip_bins", "res1"
)
ct.summary(output="perc_row")



Cross-tabs
Data     : intuit75k
Variables: zip_bins, res1
Null hyp : There is no association between zip_bins and res1
Alt. hyp : There is an association between zip_bins and res1

Row percentages:

┌──────────┬────────┬───────┬────────┐
│ zip_bins ┆ No     ┆ Yes   ┆ Total  │
╞══════════╪════════╪═══════╪════════╡
│ 1        ┆ 78.3%  ┆ 21.7% ┆ 100.0% │
│ 2        ┆ 95.8%  ┆ 4.2%  ┆ 100.0% │
│ 3        ┆ 96.63% ┆ 3.37% ┆ 100.0% │
│ 4        ┆ 95.94% ┆ 4.06% ┆ 100.0% │
│ 5        ┆ 96.65% ┆ 3.35% ┆ 100.0% │
│ 6        ┆ 96.23% ┆ 3.77% ┆ 100.0% │
│ 7        ┆ 96.48% ┆ 3.52% ┆ 100.0% │
│ 8        ┆ 96.21% ┆ 3.79% ┆ 100.0% │
│ 9        ┆ 96.32% ┆ 3.68% ┆ 100.0% │
│ 10       ┆ 96.45% ┆ 3.55% ┆ 100.0% │
│ 11       ┆ 96.3%  ┆ 3.7%  ┆ 100.0% │
│ 12       ┆ 95.13% ┆ 4.87% ┆ 100.0% │
│ 13       ┆ 96.73% ┆ 3.27% ┆ 100.0% │
│ 14       ┆ 96.17% ┆ 3.83% ┆ 100.0% │
│ 15       ┆ 96.55% ┆ 3.45% ┆ 100.0% │
│ 16       ┆ 96.18% ┆ 3.82% ┆ 100.0% │
│ 17       ┆ 96.12% ┆ 3.88% ┆ 100.0% │
│ 18       ┆ 94.79% 

Gender identity and resp1

In [7]:
ct = rsm.basics.cross_tabs(
    {"intuit75k": intuit75k.filter(pl.col("training") == 1)}, "sex", "res1"
)
ct.summary(output="perc_row")



Cross-tabs
Data     : intuit75k
Variables: sex, res1
Null hyp : There is no association between sex and res1
Alt. hyp : There is an association between sex and res1

Row percentages:

┌─────────┬────────┬───────┬────────┐
│ sex     ┆ No     ┆ Yes   ┆ Total  │
╞═════════╪════════╪═══════╪════════╡
│ Female  ┆ 95.2%  ┆ 4.8%  ┆ 100.0% │
│ Male    ┆ 95.24% ┆ 4.76% ┆ 100.0% │
│ Unknown ┆ 95.3%  ┆ 4.7%  ┆ 100.0% │
│ Total   ┆ 95.24% ┆ 4.76% ┆ 100.0% │
└─────────┴────────┴───────┴────────┘

Chi-squared: 0.1 df(2), p.value 0.95
0.0% of cells have expected values below 5



Bizflag

In [8]:
ct = rsm.basics.cross_tabs(
    {"intuit75k": intuit75k.filter(pl.col("training") == 1)}, "bizflag", "res1"
)
ct.summary(output="perc_row")



Cross-tabs
Data     : intuit75k
Variables: bizflag, res1
Null hyp : There is no association between bizflag and res1
Alt. hyp : There is an association between bizflag and res1

Row percentages:

┌─────────┬────────┬───────┬────────┐
│ bizflag ┆ No     ┆ Yes   ┆ Total  │
╞═════════╪════════╪═══════╪════════╡
│ 0       ┆ 95.29% ┆ 4.71% ┆ 100.0% │
│ 1       ┆ 95.08% ┆ 4.92% ┆ 100.0% │
│ Total   ┆ 95.24% ┆ 4.76% ┆ 100.0% │
└─────────┴────────┴───────┴────────┘

Chi-squared: 1.01 df(1), p.value 0.32
0.0% of cells have expected values below 5



Version1

In [9]:
ct = rsm.basics.cross_tabs(
    {"intuit75k": intuit75k.filter(pl.col("training") == 1)}, "version1", "res1"
)
ct.summary(output="perc_row")



Cross-tabs
Data     : intuit75k
Variables: version1, res1
Null hyp : There is no association between version1 and res1
Alt. hyp : There is an association between version1 and res1

Row percentages:

┌──────────┬────────┬───────┬────────┐
│ version1 ┆ No     ┆ Yes   ┆ Total  │
╞══════════╪════════╪═══════╪════════╡
│ 0        ┆ 95.67% ┆ 4.33% ┆ 100.0% │
│ 1        ┆ 93.65% ┆ 6.35% ┆ 100.0% │
│ Total    ┆ 95.24% ┆ 4.76% ┆ 100.0% │
└──────────┴────────┴───────┴────────┘

Chi-squared: 79.91 df(1), p.value < .001
0.0% of cells have expected values below 5



Owntaxprod

In [10]:
ct = rsm.basics.cross_tabs(
    {"intuit75k": intuit75k.filter(pl.col("training") == 1)}, "owntaxprod", "res1"
)
ct.summary(output="perc_row")



Cross-tabs
Data     : intuit75k
Variables: owntaxprod, res1
Null hyp : There is no association between owntaxprod and res1
Alt. hyp : There is an association between owntaxprod and res1

Row percentages:

┌────────────┬────────┬───────┬────────┐
│ owntaxprod ┆ No     ┆ Yes   ┆ Total  │
╞════════════╪════════╪═══════╪════════╡
│ 0          ┆ 95.34% ┆ 4.66% ┆ 100.0% │
│ 1          ┆ 91.82% ┆ 8.18% ┆ 100.0% │
│ Total      ┆ 95.24% ┆ 4.76% ┆ 100.0% │
└────────────┴────────┴───────┴────────┘

Chi-squared: 40.07 df(1), p.value < .001
0.0% of cells have expected values below 5



Upgraded

In [11]:
rsm.eda.pivot(intuit75k, "version1", "upgraded")


version1,0,1
i32,f64,f64
0,43321.0,15629.0
1,16050.0,null


In [12]:
ct = rsm.basics.cross_tabs(
    {"intuit75k": intuit75k.filter(pl.col("training") == 1)}, "upgraded", "res1"
)
ct.summary(output="perc_row")



Cross-tabs
Data     : intuit75k
Variables: upgraded, res1
Null hyp : There is no association between upgraded and res1
Alt. hyp : There is an association between upgraded and res1

Row percentages:

┌──────────┬────────┬───────┬────────┐
│ upgraded ┆ No     ┆ Yes   ┆ Total  │
╞══════════╪════════╪═══════╪════════╡
│ 0        ┆ 96.01% ┆ 3.99% ┆ 100.0% │
│ 1        ┆ 92.3%  ┆ 7.7%  ┆ 100.0% │
│ Total    ┆ 95.24% ┆ 4.76% ┆ 100.0% │
└──────────┴────────┴───────┴────────┘

Chi-squared: 262.79 df(1), p.value < .001
0.0% of cells have expected values below 5



## Early Findings From Descriptive Analysis

The cross-tabs help identify the first business signals before any model is fit. Variables like prior upgrade history, current version, tax-product ownership, and purchasing behavior appear meaningfully associated with response, which is consistent with an upsell story: customers already engaged with the Intuit ecosystem are more likely to upgrade again. These descriptive results do not define the final mailing rule, but they justify building predictive models around customer relationship depth, product maturity, and recency.


Numords

In [13]:
reg = rsm.model.regress(
    data={"intuit75k": intuit75k.filter(pl.col("training") == 1)},
    rvar="numords",
    evar="res1",
)

reg.coef.with_columns(cs.numeric().round(3))


index,coefficient,std.error,t.value,p.value,
str,f64,f64,f64,f64,str
"""Intercept""",2.046,0.006,370.514,0.0,"""***"""
"""res1[Yes]""",0.523,0.025,20.643,0.0,"""***"""


Dollars

In [14]:
reg = rsm.model.regress(
    data={"intuit75k": intuit75k.filter(pl.col("training") == 1)},
    rvar="dollars",
    evar="res1",
)

reg.coef.with_columns(cs.numeric().round(3))


index,coefficient,std.error,t.value,p.value,
str,f64,f64,f64,f64,str
"""Intercept""",91.467,0.361,253.399,0.0,"""***"""
"""res1[Yes]""",25.533,1.655,15.43,0.0,"""***"""


Last

In [15]:
reg = rsm.model.regress(
    data={"intuit75k": intuit75k.filter(pl.col("training") == 1)},
    rvar="last",
    evar="res1",
)

reg.coef.with_columns(cs.numeric().round(3))

index,coefficient,std.error,t.value,p.value,
str,f64,f64,f64,f64,str
"""Intercept""",16.048,0.043,377.404,0.0,"""***"""
"""res1[Yes]""",-4.026,0.195,-20.653,0.0,"""***"""


Sincepurch

In [16]:
reg = rsm.model.regress(
    data={"intuit75k": intuit75k.filter(pl.col("training") == 1)},
    rvar="sincepurch",
    evar="res1",
)

reg.coef.with_columns(cs.numeric().round(3))


index,coefficient,std.error,t.value,p.value,
str,f64,f64,f64,f64,str
"""Intercept""",15.44,0.045,345.505,0.0,"""***"""
"""res1[Yes]""",3.712,0.205,18.121,0.0,"""***"""


### **Logistic Regression Estimation**

In [17]:
clf = rsm.model.logistic(
    data={"intuit75k": intuit75k.filter(pl.col("training") == 1)},
    rvar="res1",
    lev="Yes",
    evar=[
        "zip_bins",
        "sex",
        "bizflag",
        "numords",
        "dollars",
        "last",
        "sincepurch",
        "version1",
        "owntaxprod",
        "upgraded",
    ],
)

clf.coef.with_columns(cs.numeric().round(3))

index,OR,OR%,coefficient,std.error,z.value,p.value,
str,f64,f64,f64,f64,f64,f64,str
"""Intercept""",0.051,-94.881,-2.972,0.079,-37.701,0.0,"""***"""
"""sex[Female]""",1.013,1.302,0.013,0.052,0.249,0.804,""" """
"""sex[Unknown]""",1.004,0.421,0.004,0.064,0.066,0.948,""" """
"""zip_bins""",0.946,-5.376,-0.055,0.004,-14.931,0.0,"""***"""
"""bizflag""",1.047,4.743,0.046,0.048,0.964,0.335,""" """
"""numords""",1.251,25.15,0.224,0.019,12.01,0.0,"""***"""
"""dollars""",1.001,0.097,0.001,0.0,3.638,0.0,"""***"""
"""last""",0.958,-4.172,-0.043,0.002,-17.959,0.0,"""***"""
"""sincepurch""",1.002,0.203,0.002,0.004,0.518,0.605,""" """


## Logistic Regression Interpretation

The baseline logistic model gives a useful business benchmark because its coefficients can be read directionally: more engagement, more prior orders, and stronger product ownership signals should generally raise response probability, while weaker or older customer relationships should lower it. Even if more flexible models later outperform it, logistic regression remains valuable because it anchors the story in interpretable drivers rather than black-box ranking alone.


Zip_bins as a categorical value

In [18]:
train = intuit75k.filter(pl.col("training") == 1).with_columns(
    pl.col("zip_bins").cast(pl.Utf8).cast(pl.Categorical).alias("zip_bins")
)

# build the desired level order: "1","2","3",... based on the labels
levels = (
    train.select(pl.col("zip_bins").cast(pl.Utf8).alias("zb"))
    .unique()
    .with_columns(pl.col("zb").cast(pl.Int32).alias("zb_num"))
    .sort("zb_num")
    .select("zb")
    .to_series()
    .to_list()
)

# force that order (same labels, but now "1" is first => intercept)
train = train.with_columns(
    pl.col("zip_bins").cast(pl.Utf8).cast(pl.Enum(levels)).alias("zip_bins")
)

clf_base = rsm.model.logistic(
    data={"intuit75k": train},
    rvar="res1",
    lev="Yes",
    evar=[
        "zip_bins",
        "sex",
        "bizflag",
        "numords",
        "dollars",
        "last",
        "sincepurch",
        "version1",
        "owntaxprod",
        "upgraded",
    ],
)

clf_base.coef


index,OR,OR%,coefficient,std.error,z.value,p.value,
str,f64,f64,f64,f64,f64,f64,str
"""Intercept""",0.176214,-82.378581,-1.736055,0.084464,-20.553828,7.1127e-94,"""***"""
"""zip_bins[2]""",0.148077,-85.192253,-1.91002,0.110428,-17.296484,4.9999e-67,"""***"""
"""zip_bins[3]""",0.117537,-88.246305,-2.141003,0.120893,-17.709905,3.5167e-70,"""***"""
"""zip_bins[4]""",0.13585,-86.415022,-1.996206,0.11208,-17.810522,5.8561e-71,"""***"""
"""zip_bins[5]""",0.116276,-88.372353,-2.151785,0.120254,-17.893636,1.3219e-71,"""***"""
"""zip_bins[6]""",0.126497,-87.350256,-2.067533,0.114591,-18.042701,9.0034e-73,"""***"""
"""zip_bins[7]""",0.121261,-87.873916,-2.109811,0.117407,-17.970028,3.3455e-72,"""***"""
"""zip_bins[8]""",0.130754,-86.924576,-2.034436,0.114015,-17.843645,3.2386e-71,"""***"""
"""zip_bins[9]""",0.124183,-87.581653,-2.085995,0.116823,-17.855979,2.5969e-71,"""***"""


In [19]:
clf_base.plot(plots="pred", incl=["zip_bins"])


In [20]:
clf_base.plot("pip")

In [21]:
(
    train.filter((pl.col("training") == 1) & (pl.col("zip_bins") == "1"))
    .group_by("zip5")
    .agg(
        pl.col("res1_yes").mean().alias("mean"),
        pl.col("res1_yes").sum().alias("sum"),
        pl.col("res1_yes").count().alias("count"),
    )
    .sort("sum", descending=True)
    .head(10)
    .with_columns(pl.format("{}%", (pl.col("mean") * 100).round(2).alias("mean")))
)

zip5,mean,sum,count
str,str,i64,u32
"""00801""","""41.12%""",486,1182
"""00804""","""35.43%""",45,127
"""00000""","""3.0%""",3,100
"""01890""","""17.65%""",3,17
"""01923""","""37.5%""",3,8
"""01863""","""22.22%""",2,9
"""02050""","""28.57%""",2,7
"""01950""","""50.0%""",2,4
"""01752""","""13.33%""",2,15


Improving model by adding dummies for zip 00801 and 00804

In [22]:
train = train.with_columns(
    [
        pl.when(pl.col("zip5") == "00801").then(1).otherwise(0).alias("zip801"),
        pl.when(pl.col("zip5") == "00804").then(1).otherwise(0).alias("zip804"),
    ]
)

clf = rsm.model.logistic(
    data={"intuit75k": train.filter(pl.col("training") == 1)},
    rvar="res1",
    lev="Yes",
    evar=[
        "zip_bins",
        "sex",
        "bizflag",
        "numords",
        "dollars",
        "last",
        "sincepurch",
        "version1",
        "owntaxprod",
        "upgraded",
        "zip801",
        "zip804",
    ],
)
clf.coef[1:20].with_columns(cs.numeric().round(3))

index,OR,OR%,coefficient,std.error,z.value,p.value,
str,f64,f64,f64,f64,f64,f64,str
"""zip_bins[2]""",1.3,30.004,0.262,0.18,1.457,0.145,""" """
"""zip_bins[3]""",1.031,3.127,0.031,0.187,0.165,0.869,""" """
"""zip_bins[4]""",1.189,18.893,0.173,0.181,0.956,0.339,""" """
"""zip_bins[5]""",1.02,2.047,0.02,0.186,0.109,0.913,""" """
"""zip_bins[6]""",1.107,10.702,0.102,0.183,0.557,0.578,""" """
"""zip_bins[7]""",1.063,6.292,0.061,0.184,0.331,0.741,""" """
"""zip_bins[8]""",1.147,14.668,0.137,0.182,0.751,0.453,""" """
"""zip_bins[9]""",1.088,8.847,0.085,0.184,0.461,0.645,""" """
"""zip_bins[10]""",1.063,6.349,0.062,0.185,0.334,0.739,""" """


In [23]:
clf.summary(test="zip_bins")

Logistic regression (GLM)
Data                 : intuit75k
Response variable    : res1
Level                : Yes
Explanatory variables: zip_bins, sex, bizflag, numords, dollars, last, sincepurch, version1, owntaxprod, upgraded, zip801, zip804
Null hyp.: There is no effect of x on res1
Alt. hyp.: There is an effect of x on res1

┌──────────────┬────────┬─────────┬─────────────┬───────────┬─────────┬─────────┬─────┐
│ index        ┆ OR     ┆ OR%     ┆ coefficient ┆ std.error ┆ z.value ┆ p.value ┆     │
╞══════════════╪════════╪═════════╪═════════════╪═══════════╪═════════╪═════════╪═════╡
│ Intercept    ┆ 0.019  ┆ -98.1%  ┆ -3.967      ┆ 0.168     ┆ -23.678 ┆ < .001  ┆ *** │
│ zip_bins[2]  ┆ 1.3    ┆ 30.0%   ┆ 0.262       ┆ 0.18      ┆ 1.457   ┆ 0.145   ┆     │
│ zip_bins[3]  ┆ 1.031  ┆ 3.1%    ┆ 0.031       ┆ 0.187     ┆ 0.165   ┆ 0.869   ┆     │
│ zip_bins[4]  ┆ 1.189  ┆ 18.9%   ┆ 0.173       ┆ 0.181     ┆ 0.956   ┆ 0.339   ┆     │
│ zip_bins[5]  ┆ 1.02   ┆ 2.0%    ┆ 0.02        ┆ 0.1

## ZIP and Interaction Insight

Adding targeted ZIP effects and interaction structure acknowledges that not all customers respond in the same way. In business terms, this means response depends not just on who the customer is, but on how combinations of geography, product status, and purchase history interact. That is important for marketing because a single mailing rule applied uniformly across the file can leave money on the table if important subgroup effects are ignored.


### **Estimating Neural Network (MLP)**

In [24]:
clf1 = rsm.model.mlp(
    data={"intuit75k": train.filter(pl.col("training") == 1)},
    rvar="res1",
    lev="Yes",
    evar=[
        "zip_bins",
        "sex",
        "bizflag",
        "numords",
        "dollars",
        "last",
        "sincepurch",
        "version1",
        "owntaxprod",
        "upgraded",
        "zip801",
        "zip804",
    ],
    hidden_layer_sizes=(1,),
    alpha=0.001,
    max_iter=100_000,
    random_state=1234,
)
clf1.summary()

Multi-layer Perceptron (NN)
Data                 : intuit75k
Response variable    : res1
Level                : Yes
Explanatory variables: zip_bins, sex, bizflag, numords, dollars, last, sincepurch, version1, owntaxprod, upgraded, zip801, zip804
Model type           : classification
Nr. of features      : (12, 31)
Nr. of weights       : 34
Nr. of observations  : 52,500
Hidden_layer_sizes   : (1,)
Activation function  : tanh
Solver               : lbfgs
Alpha                : 0.001
Batch size           : auto
Learning rate        : 0.001
Maximum iterations   : 100000
random_state         : 1234
AUC                  : 0.768

Raw data             :
shape: (5, 12)
┌──────────┬──────┬─────────┬─────────┬─────────┬──────┬────────────┬──────────┬────────────┬──────────┬────────┬────────┐
│ zip_bins ┆ sex  ┆ bizflag ┆ numords ┆ dollars ┆ last ┆ sincepurch ┆ version1 ┆ owntaxprod ┆ upgraded ┆ zip801 ┆ zip804 │
│ ---      ┆ ---  ┆ ---     ┆ ---     ┆ ---     ┆ ---  ┆ ---        ┆ ---      ┆ --- 

In [25]:
clf_cv = rsm.model.mlp(
    data={"intuit75k": train.filter(pl.col("training") == 1)},
    rvar="res1",
    lev="Yes",
    evar=[
        "zip_bins",
        "sex",
        "bizflag",
        "numords",
        "dollars",
        "last",
        "sincepurch",
        "version1",
        "owntaxprod",
        "upgraded",
        "zip801",
        "zip804",
    ],
    hidden_layer_sizes=(5,),
    random_state=1234,
    cv=5,
)

clf_cv.summary()


Multi-layer Perceptron (NN)
Data                 : intuit75k
Response variable    : res1
Level                : Yes
Explanatory variables: zip_bins, sex, bizflag, numords, dollars, last, sincepurch, version1, owntaxprod, upgraded, zip801, zip804
Model type           : classification
Nr. of features      : (12, 31)
Nr. of weights       : 166
Nr. of observations  : 52,500
Hidden_layer_sizes   : (5,)
Activation function  : tanh
Solver               : lbfgs
Alpha                : 0.0001
Batch size           : auto
Learning rate        : 0.001
Maximum iterations   : 1000000
random_state         : 1234
AUC                  : 0.784

Raw data             :
shape: (5, 12)
┌──────────┬──────┬─────────┬─────────┬─────────┬──────┬────────────┬──────────┬────────────┬──────────┬────────┬────────┐
│ zip_bins ┆ sex  ┆ bizflag ┆ numords ┆ dollars ┆ last ┆ sincepurch ┆ version1 ┆ owntaxprod ┆ upgraded ┆ zip801 ┆ zip804 │
│ ---      ┆ ---  ┆ ---     ┆ ---     ┆ ---     ┆ ---  ┆ ---        ┆ ---      ┆ -

In [26]:
clf1.plot("pip_sklearn")

In [27]:
clf_cv.plot("pip_sklearn")


In [28]:
ret = clf1.plot("pip", ret=True)
ret

(<plotnine.ggplot.ggplot object at 0x3797499d0>,
 shape: (12, 2)
 ┌────────────┬────────────┐
 │ variable   ┆ importance │
 │ ---        ┆ ---        │
 │ str        ┆ f64        │
 ╞════════════╪════════════╡
 │ zip801     ┆ 0.081733   │
 │ upgraded   ┆ 0.056448   │
 │ last       ┆ 0.040696   │
 │ version1   ┆ 0.023291   │
 │ numords    ┆ 0.0232     │
 │ zip804     ┆ 0.006441   │
 │ zip_bins   ┆ 0.002551   │
 │ owntaxprod ┆ 0.001465   │
 │ dollars    ┆ 0.001289   │
 │ bizflag    ┆ 0.000224   │
 │ sex        ┆ 0.000117   │
 │ sincepurch ┆ 0.000033   │
 └────────────┴────────────┘)

In [29]:
ret = clf_cv.plot("pip", ret=True)
ret


(<plotnine.ggplot.ggplot object at 0x37974bb90>,
 shape: (12, 2)
 ┌────────────┬────────────┐
 │ variable   ┆ importance │
 │ ---        ┆ ---        │
 │ str        ┆ f64        │
 ╞════════════╪════════════╡
 │ zip801     ┆ 0.08654    │
 │ upgraded   ┆ 0.055118   │
 │ last       ┆ 0.049329   │
 │ numords    ┆ 0.03733    │
 │ version1   ┆ 0.02089    │
 │ zip_bins   ┆ 0.018772   │
 │ zip804     ┆ 0.006385   │
 │ sincepurch ┆ 0.005466   │
 │ dollars    ┆ 0.005168   │
 │ owntaxprod ┆ 0.003519   │
 │ bizflag    ┆ 0.001926   │
 │ sex        ┆ 0.001787   │
 └────────────┴────────────┘)

Comparing neural networks

In [30]:
clf1.summary()
clf_cv.summary()


Multi-layer Perceptron (NN)
Data                 : intuit75k
Response variable    : res1
Level                : Yes
Explanatory variables: zip_bins, sex, bizflag, numords, dollars, last, sincepurch, version1, owntaxprod, upgraded, zip801, zip804
Model type           : classification
Nr. of features      : (12, 31)
Nr. of weights       : 34
Nr. of observations  : 52,500
Hidden_layer_sizes   : (1,)
Activation function  : tanh
Solver               : lbfgs
Alpha                : 0.001
Batch size           : auto
Learning rate        : 0.001
Maximum iterations   : 100000
random_state         : 1234
AUC                  : 0.768

Raw data             :
shape: (5, 12)
┌──────────┬──────┬─────────┬─────────┬─────────┬──────┬────────────┬──────────┬────────────┬──────────┬────────┬────────┐
│ zip_bins ┆ sex  ┆ bizflag ┆ numords ┆ dollars ┆ last ┆ sincepurch ┆ version1 ┆ owntaxprod ┆ upgraded ┆ zip801 ┆ zip804 │
│ ---      ┆ ---  ┆ ---     ┆ ---     ┆ ---     ┆ ---  ┆ ---        ┆ ---      ┆ --- 

## Neural Network Comparison

The neural-network models test whether nonlinear structure improves ranking quality over the simpler statistical baseline. The business interpretation is straightforward: if response behavior is driven by threshold effects or combinations of signals that logistic regression cannot capture cleanly, a flexible model should identify more profitable customers near the top of the mailing list.


### **Interactions**

In [31]:
clf_cv.plot("pdp", incl=[], incl_int=["last:version1", "numords:version1"])

<Figure size 800x300 with 1 Axes>

In [32]:
clf_cv.plot("pdp", incl=[], incl_int=["last:dollars", "last:upgraded"])


<Figure size 800x300 with 1 Axes>

In [33]:
clf = rsm.model.logistic(
    data={"intuit75k": train.filter(pl.col("training") == 1)},
    rvar="res1",
    lev="Yes",
    evar=[
        "zip_bins",
        "sex",
        "bizflag",
        "numords",
        "dollars",
        "last",
        "sincepurch",
        "version1",
        "owntaxprod",
        "upgraded",
        "zip801",
        "zip804",
    ],
    ivar=["numords:version1", "last:version1"],
)
rows = [23, 26, 27, 32, 33]

pl.concat([clf.coef.slice(i, 1) for i in rows]).with_columns(
    pl.selectors.numeric().round(3)
)


index,OR,OR%,coefficient,std.error,z.value,p.value,
str,f64,f64,f64,f64,f64,f64,str
"""numords""",1.162,16.195,0.15,0.022,6.7,0.0,"""***"""
"""sincepurch""",1.002,0.218,0.002,0.004,0.531,0.595,""" """
"""version1""",1.474,47.397,0.388,0.151,2.572,0.01,"""*"""
"""numords:version1""",1.397,39.716,0.334,0.035,9.442,0.0,"""***"""
"""last:version1""",0.959,-4.073,-0.042,0.006,-7.089,0.0,"""***"""


In [34]:
clf.plot("pdp", incl=[], incl_int=["last:version1", "numords:version1"])

<Figure size 800x300 with 1 Axes>

In [35]:
clf.summary()

Logistic regression (GLM)
Data                 : intuit75k
Response variable    : res1
Level                : Yes
Explanatory variables: zip_bins, sex, bizflag, numords, dollars, last, sincepurch, version1, owntaxprod, upgraded, zip801, zip804
Null hyp.: There is no effect of x on res1
Alt. hyp.: There is an effect of x on res1

┌──────────────────┬────────┬─────────┬─────────────┬───────────┬─────────┬─────────┬─────┐
│ index            ┆ OR     ┆ OR%     ┆ coefficient ┆ std.error ┆ z.value ┆ p.value ┆     │
╞══════════════════╪════════╪═════════╪═════════════╪═══════════╪═════════╪═════════╪═════╡
│ Intercept        ┆ 0.021  ┆ -97.9%  ┆ -3.86       ┆ 0.171     ┆ -22.541 ┆ < .001  ┆ *** │
│ zip_bins[2]      ┆ 1.296  ┆ 29.6%   ┆ 0.259       ┆ 0.181     ┆ 1.436   ┆ 0.151   ┆     │
│ zip_bins[3]      ┆ 1.034  ┆ 3.4%    ┆ 0.034       ┆ 0.187     ┆ 0.181   ┆ 0.857   ┆     │
│ zip_bins[4]      ┆ 1.18   ┆ 18.0%   ┆ 0.166       ┆ 0.182     ┆ 0.912   ┆ 0.362   ┆     │
│ zip_bins[5]      ┆ 1.01

## Interaction Model Interpretation

The interaction analysis shows that response likelihood is not purely additive. Variables such as recency, order volume, software version, and spending appear to reinforce or weaken each other. For decision-making, that means Intuit should think in terms of **customer profiles** rather than isolated variables: some combinations of traits represent much stronger upgrade opportunity than any single feature alone.


### **Cross Validation for Tuning the ML Model**

In [36]:
hls = [[i, j] for i in (1, 3, 5) for j in (1, 3, 5)]

param_grid = {
    "hidden_layer_sizes": [tuple(x) for x in hls],
    "alpha": [1e-5, 1e-4, 1e-3, 1e-2, 0.1, 0.5],
}


In [37]:
X = clf1.data_onehot
y = clf1.data["res1"].to_numpy().ravel()

cv = GridSearchCV(
    estimator=clf1.fitted,
    param_grid=param_grid,
    scoring={"AUC": "roc_auc"},
    refit="AUC",
    cv=5,
    n_jobs=1,
    verbose=0,
)

cv.fit(X, y)

cv.best_params_, cv.best_score_


({'alpha': 0.5, 'hidden_layer_sizes': (3, 3)}, np.float64(0.7664503797302193))

In [38]:
stratified_kfolds = StratifiedKFold(n_splits=5, shuffle=True, random_state=1234)
X = clf1.data_onehot
y = clf1.data["res1"].to_numpy().ravel()

cv1 = GridSearchCV(
    estimator=clf1.fitted,
    param_grid=param_grid,
    scoring={"AUC": "roc_auc"},
    refit="AUC",
    cv=stratified_kfolds,
    n_jobs=1,
    verbose=0,
)

cv1.fit(X, y)

cv1.best_params_, cv1.best_score_

({'alpha': 0.5, 'hidden_layer_sizes': (3, 5)}, np.float64(0.7682803678594825))

In [39]:
clf_cv = rsm.model.mlp(
    data={"intuit75k": train.filter(pl.col("training") == 1)},
    rvar="res1",
    lev="Yes",
    evar=[
        "zip_bins",
        "sex",
        "bizflag",
        "numords",
        "dollars",
        "last",
        "sincepurch",
        "version1",
        "owntaxprod",
        "upgraded",
        "zip801",
        "zip804",
    ],
    random_state=1234,
    **cv1.best_params_,
)

clf_cv.summary()


Multi-layer Perceptron (NN)
Data                 : intuit75k
Response variable    : res1
Level                : Yes
Explanatory variables: zip_bins, sex, bizflag, numords, dollars, last, sincepurch, version1, owntaxprod, upgraded, zip801, zip804
Model type           : classification
Nr. of features      : (12, 31)
Nr. of weights       : 122
Nr. of observations  : 52,500
Hidden_layer_sizes   : (3, 5)
Activation function  : tanh
Solver               : lbfgs
Alpha                : 0.5
Batch size           : auto
Learning rate        : 0.001
Maximum iterations   : 1000000
random_state         : 1234
AUC                  : 0.781

Raw data             :
shape: (5, 12)
┌──────────┬──────┬─────────┬─────────┬─────────┬──────┬────────────┬──────────┬────────────┬──────────┬────────┬────────┐
│ zip_bins ┆ sex  ┆ bizflag ┆ numords ┆ dollars ┆ last ┆ sincepurch ┆ version1 ┆ owntaxprod ┆ upgraded ┆ zip801 ┆ zip804 │
│ ---      ┆ ---  ┆ ---     ┆ ---     ┆ ---     ┆ ---  ┆ ---        ┆ ---      ┆ --

### **Ranking Models**

In [40]:
df_test = intuit75k.filter(pl.col("training") == 0).with_columns(
    pl.col("zip_bins").cast(pl.Utf8).cast(pl.Categorical).alias("zip_bins")
)
df_test = df_test.with_columns(
    [
        pl.when(pl.col("zip5") == "00801").then(1).otherwise(0).alias("zip801"),
        pl.when(pl.col("zip5") == "00804").then(1).otherwise(0).alias("zip804"),
    ]
)
df_mailable = df_test.filter(pl.col("res1_yes") != 1)

MAIL_COST = 1.41
MARGIN = 60.0


def profit_curve(model, df_test: pl.DataFrame):
    # filter mailable
    if "res1_yes" in df_test.columns:
        df_mailable = df_test.filter(pl.col("res1_yes") != 1)
    else:
        df_mailable = df_test.filter(pl.col("res1") != "Yes")

    pred = model.predict(data=df_mailable)
    pcol = "prediction" if "prediction" in pred.columns else pred.columns[-1]

    scored_base = pred.select(pcol).with_columns(df_mailable.select("id"))

    scored = (
        scored_base.rename({pcol: "p"})
        .sort("p", descending=True)
        .with_row_index("rank")
        .with_columns((pl.col("rank") + 1).alias("k"))
        .with_columns(pl.col("p").cum_sum().alias("exp_resp"))
        .with_columns(
            (MARGIN * pl.col("exp_resp") - MAIL_COST * pl.col("k")).alias("profit")
        )
    )

    best_row = scored.sort("profit", descending=True).head(1).to_dicts()[0]
    return scored, best_row


candidates = {"logit": clf_base, "logit_int": clf, "nn": clf1, "nn_cv": clf_cv}

best_by_model = {}
for name, m in candidates.items():
    curve, best = profit_curve(m, df_test)
    best_by_model[name] = {
        "best_k": best["k"],
        "best_profit": best["profit"],
        "exp_responders": best["exp_resp"],
    }

best_by_model


{'logit': {'best_k': 13326,
  'best_profit': 30515.175268724557,
  'exp_responders': 821.7472544787427},
 'logit_int': {'best_k': 12168,
  'best_profit': 28245.36795524753,
  'exp_responders': 756.7041325874588},
 'nn': {'best_k': 11655,
  'best_profit': 29293.376773457316,
  'exp_responders': 762.1154462242886},
 'nn_cv': {'best_k': 11627,
  'best_profit': 29439.51527991071,
  'exp_responders': 763.8930879985118}}

## Ranking Logic

At this stage the project shifts from explanation to action. The practical objective is not simply to maximize AUC in isolation, but to rank the file so that the most economically attractive customers are mailed first. That is the correct business framing for direct marketing, because once mailing cost is introduced, the key question becomes **where the profit curve peaks**.


### **Comparing to Break-Even Threshold**

In [41]:
MAIL_COST = 1.41
MARGIN = 60.0

BET = MAIL_COST / MARGIN  # break-even probability threshold
BET_adj_margin = MAIL_COST / (MARGIN / 2)
BET, BET_adj_margin


(0.0235, 0.047)

In [42]:
MAIL_COST = 1.41
MARGIN = 60.0

BET = MAIL_COST / MARGIN  # 1.41/60
BET_adj_margin = MAIL_COST / (MARGIN / 2)  # 1.41/30

best_model = clf_base

# df_test = intuit75k.filter(pl.col("training") == 0)
df_test = intuit75k.filter(pl.col("training") == 0).with_columns(
    pl.col("zip_bins").cast(pl.Utf8).cast(pl.Enum(levels)).alias("zip_bins")
)


pred = best_model.predict(data=df_test)
pcol = "prediction" if "prediction" in pred.columns else pred.columns[-1]

scored = (
    df_test.select(["id", "res1", "res1_yes"])
    .with_columns(pred.select(pl.col(pcol)).rename({pcol: "p"}))
    .with_columns(
        (pl.col("res1_yes") != 1).alias("mailable"),
        (pl.col("p") >= BET).alias("mailto_bet_eval"),
        (pl.col("p") >= BET_adj_margin).alias("mailto_bet_adjmargin_eval"),
        (pl.col("p") / 2).alias("p_half"),
        ((pl.col("p") / 2) >= BET).alias("mailto_phalf_eval"),
        ((pl.col("res1_yes") != 1) & (pl.col("p") >= BET)).alias("mailto_bet"),
        ((pl.col("res1_yes") != 1) & (pl.col("p") >= BET_adj_margin)).alias(
            "mailto_bet_adjmargin"
        ),
        ((pl.col("res1_yes") != 1) & ((pl.col("p") / 2) >= BET)).alias("mailto_phalf"),
    )
)


In [43]:
def cm_table(df: pl.DataFrame, mail_col: str) -> pl.DataFrame:
    return (
        df.with_columns(
            pl.when(pl.col("res1_yes") == 1)
            .then(pl.lit("Yes"))
            .otherwise(pl.lit("No"))
            .alias("res1_label")
        )
        .group_by([mail_col, "res1_label"])
        .len()
        .pivot(values="len", index=mail_col, on="res1_label")
        .fill_null(0)
        .sort(mail_col)
    )


cm_bet = cm_table(scored, "mailto_bet_eval")
cm_adj = cm_table(scored, "mailto_bet_adjmargin_eval")
cm_half = cm_table(scored, "mailto_phalf_eval")

cm_bet, cm_adj, cm_half


(shape: (2, 3)
 ┌─────────────────┬─────┬───────┐
 │ mailto_bet_eval ┆ Yes ┆ No    │
 │ ---             ┆ --- ┆ ---   │
 │ bool            ┆ u32 ┆ u32   │
 ╞═════════════════╪═════╪═══════╡
 │ false           ┆ 125 ┆ 8071  │
 │ true            ┆ 978 ┆ 13326 │
 └─────────────────┴─────┴───────┘,
 shape: (2, 3)
 ┌───────────────────────────┬───────┬─────┐
 │ mailto_bet_adjmargin_eval ┆ No    ┆ Yes │
 │ ---                       ┆ ---   ┆ --- │
 │ bool                      ┆ u32   ┆ u32 │
 ╞═══════════════════════════╪═══════╪═════╡
 │ false                     ┆ 15209 ┆ 354 │
 │ true                      ┆ 6188  ┆ 749 │
 └───────────────────────────┴───────┴─────┘,
 shape: (2, 3)
 ┌───────────────────┬───────┬─────┐
 │ mailto_phalf_eval ┆ No    ┆ Yes │
 │ ---               ┆ ---   ┆ --- │
 │ bool              ┆ u32   ┆ u32 │
 ╞═══════════════════╪═══════╪═════╡
 │ false             ┆ 15209 ┆ 354 │
 │ true              ┆ 6188  ┆ 749 │
 └───────────────────┴───────┴─────┘)

In [44]:
def mailed_rate(df: pl.DataFrame, mail_col: str) -> pl.DataFrame:
    return df.filter(pl.col(mail_col) == True).select(
        pl.len().alias("n_mailed"),
        pl.col("res1_yes").mean().alias("wave1_resp_rate"),
    )


rate_bet = mailed_rate(scored, "mailto_bet_eval")
rate_adj = mailed_rate(scored, "mailto_bet_adjmargin_eval")
rate_half = mailed_rate(scored, "mailto_phalf_eval")

rate_bet, rate_adj, rate_half


(shape: (1, 2)
 ┌──────────┬─────────────────┐
 │ n_mailed ┆ wave1_resp_rate │
 │ ---      ┆ ---             │
 │ u32      ┆ f64             │
 ╞══════════╪═════════════════╡
 │ 14304    ┆ 0.068372        │
 └──────────┴─────────────────┘,
 shape: (1, 2)
 ┌──────────┬─────────────────┐
 │ n_mailed ┆ wave1_resp_rate │
 │ ---      ┆ ---             │
 │ u32      ┆ f64             │
 ╞══════════╪═════════════════╡
 │ 6937     ┆ 0.107972        │
 └──────────┴─────────────────┘,
 shape: (1, 2)
 ┌──────────┬─────────────────┐
 │ n_mailed ┆ wave1_resp_rate │
 │ ---      ┆ ---             │
 │ u32      ┆ f64             │
 ╞══════════╪═════════════════╡
 │ 6937     ┆ 0.107972        │
 └──────────┴─────────────────┘)

In [45]:
# 50% rule adjustment
rate_bet = rate_bet.with_columns(
    (pl.col("wave1_resp_rate") / 2).alias("wave2_resp_rate")
)
rate_bet


n_mailed,wave1_resp_rate,wave2_resp_rate
u32,f64,f64
14304,0.068372,0.034186


### Scaling to full population

In [46]:
N_TOTAL = 801_821
N_RESP = 38_487
N_WAVE2 = N_TOTAL - N_RESP  # 763,334


In [47]:
def scale_profit(df: pl.DataFrame, mail_eval_col: str) -> pl.DataFrame:
    mailed_share = df.select(pl.col(mail_eval_col).mean().alias("share_mailed")).item()

    # wave1 response among mailed
    wave1_rate = (
        df.filter(pl.col(mail_eval_col) == True)
        .select(pl.col("res1_yes").mean())
        .item()
    )

    # apply 50% rule
    wave2_rate = wave1_rate / 2

    # scale
    n_mailed_wave2 = N_WAVE2 * mailed_share
    exp_buyers = n_mailed_wave2 * wave2_rate
    mailing_cost = n_mailed_wave2 * MAIL_COST
    revenue = exp_buyers * MARGIN
    profit = revenue - mailing_cost
    rome = profit / mailing_cost

    return pl.DataFrame(
        [
            {
                "rule": mail_eval_col,
                "share_mailed": mailed_share,
                "wave1_rate": wave1_rate,
                "wave2_rate": wave2_rate,
                "n_mailed_wave2": n_mailed_wave2,
                "exp_buyers": exp_buyers,
                "profit": profit,
                "rome": rome,
            }
        ]
    )


scaled = pl.concat(
    [
        scale_profit(scored, "mailto_bet_eval"),
        scale_profit(scored, "mailto_bet_adjmargin_eval"),
        scale_profit(scored, "mailto_phalf_eval"),
    ]
)

scaled


rule,share_mailed,wave1_rate,wave2_rate,n_mailed_wave2,exp_buyers,profit,rome
str,f64,f64,f64,f64,f64,f64,f64
"""mailto_bet_eval""",0.635733,0.068372,0.034186,485276.868267,16589.792267,311147.151744,0.454734
"""mailto_bet_adjmargin_eval""",0.308311,0.107972,0.053986,235344.353689,12705.270356,430480.682632,1.297271
"""mailto_phalf_eval""",0.308311,0.107972,0.053986,235344.353689,12705.270356,430480.682632,1.297271


In [48]:
def scale_to_population(
    scored: pl.DataFrame, share_col: str, rate_col: str, label: str
) -> pl.DataFrame:
    share = scored.select(pl.col(share_col).mean()).item()

    # wave1 response rate among mailed
    wave1_rate = (
        scored.filter(pl.col(rate_col) == True).select(pl.col("res1_yes").mean()).item()
    )

    wave2_rate = wave1_rate / 2  # 50% rule

    n_mailed_wave2 = N_WAVE2 * share
    exp_buyers = n_mailed_wave2 * wave2_rate

    cost = MAIL_COST * n_mailed_wave2
    profit = MARGIN * exp_buyers - cost
    rome = profit / cost

    return pl.DataFrame(
        [
            {
                "rule": label,
                "share_mailed": share,
                "wave1_rate": wave1_rate,
                "wave2_rate": wave2_rate,
                "n_mailed_wave2": n_mailed_wave2,
                "exp_buyers": exp_buyers,
                "cost": cost,
                "profit": profit,
                "rome": rome,
            }
        ]
    )


In [49]:
scaled = pl.concat(
    [
        scale_to_population(scored, "mailto_bet_eval", "mailto_bet_eval", "BET (eval)"),
        scale_to_population(
            scored,
            "mailto_bet_adjmargin_eval",
            "mailto_bet_adjmargin_eval",
            "AdjMargin (eval)",
        ),
        scale_to_population(
            scored, "mailto_phalf_eval", "mailto_phalf_eval", "p/2 (eval)"
        ),
        scale_to_population(
            scored, "mailto_bet", "mailto_bet_eval", "BET (deploy volume, eval rate)"
        ),
        scale_to_population(
            scored,
            "mailto_bet_adjmargin",
            "mailto_bet_adjmargin_eval",
            "AdjMargin (deploy volume, eval rate)",
        ),
        scale_to_population(
            scored,
            "mailto_phalf",
            "mailto_phalf_eval",
            "p/2 (deploy volume, eval rate)",
        ),
    ]
).sort("profit", descending=True)

scaled


rule,share_mailed,wave1_rate,wave2_rate,n_mailed_wave2,exp_buyers,cost,profit,rome
str,f64,f64,f64,f64,f64,f64,f64,f64
"""AdjMargin (eval)""",0.308311,0.107972,0.053986,235344.353689,12705.270356,331835.538701,430480.682632,1.297271
"""p/2 (eval)""",0.308311,0.107972,0.053986,235344.353689,12705.270356,331835.538701,430480.682632,1.297271
"""AdjMargin (deploy volume, eval rate)""",0.275022,0.107972,0.053986,209933.812978,11333.460136,296006.676299,384000.931833,1.297271
"""p/2 (deploy volume, eval rate)""",0.275022,0.107972,0.053986,209933.812978,11333.460136,296006.676299,384000.931833,1.297271
"""BET (eval)""",0.635733,0.068372,0.034186,485276.868267,16589.792267,684240.384256,311147.151744,0.454734
"""BET (deploy volume, eval rate)""",0.592267,0.068372,0.034186,452097.283733,15455.506973,637457.170064,289873.248332,0.454734


In [50]:
df_plot = (
    scaled.select(["rule", "profit"])
    .with_columns(pl.col("profit").round(0).cast(pl.Int64))
    .rename({"rule": "model"})
    .to_pandas()
)

p = (
    ggplot(df_plot, aes(x="model", y="profit"))
    + geom_col()
    + geom_text(aes(label="profit"), va="bottom", format_string="{:.0f}", size=8)
    + theme_minimal()
    + labs(title="Campaign profit", x="", y="Profit")
)

p


## Benchmark Strategy Interpretation

The benchmark rules show that simple heuristics can already create substantial value when the file is large enough, but they also establish the baseline that a modeling approach must beat. In business terms, this is the minimum standard for deployment: a more complex model is only worth operationalizing if it improves profit or marketing efficiency relative to easier rules such as break-even or adjusted-margin thresholds.


# INTUIT REDUX

### **Creating and tuning new models**

### Logistic Regression

In [51]:
# Loading data
intuit75k = pl.read_parquet("data/intuit75k.parquet")
intuit75k = intuit75k.drop("res1_yes")
intuit75k = intuit75k.with_columns(
    [
        pl.col("zip_bins").cast(pl.Utf8).cast(pl.Categorical).alias("zip_bins"),
        (pl.col("zip5") == "00801").cast(pl.Int8).alias("zip801"),
        (pl.col("zip5") == "00804").cast(pl.Int8).alias("zip804"),
    ]
)

intuit75k.head()

id,zip5,zip_bins,sex,bizflag,numords,dollars,last,sincepurch,version1,owntaxprod,upgraded,res1,training,zip801,zip804
i32,str,cat,cat,i32,i32,f64,i32,i32,i32,i32,i32,cat,i32,i8,i8
1,"""94553""","""18""","""Male""",0,2,109.5,5,12,0,0,0,"""No""",1,0,0
2,"""53190""","""10""","""Unknown""",0,1,69.5,4,3,0,0,0,"""No""",0,0,0
3,"""37091""","""8""","""Male""",0,4,93.0,14,29,0,0,1,"""No""",0,0,0
4,"""02125""","""1""","""Male""",0,1,22.0,17,1,0,0,0,"""No""",1,0,0
5,"""60201""","""11""","""Male""",0,1,24.5,2,3,0,0,0,"""No""",0,0,0


In [52]:
clf = rsm.model.logistic(
    data={"intuit75k": intuit75k.filter(pl.col("training") == 1)},
    rvar="res1",
    lev="Yes",
    evar=[
        "zip_bins",
        "sex",
        "bizflag",
        "numords",
        "dollars",
        "last",
        "sincepurch",
        "version1",
        "owntaxprod",
        "upgraded",
        "zip801",
        "zip804",
    ],
)

clf.summary()


Logistic regression (GLM)
Data                 : intuit75k
Response variable    : res1
Level                : Yes
Explanatory variables: zip_bins, sex, bizflag, numords, dollars, last, sincepurch, version1, owntaxprod, upgraded, zip801, zip804
Null hyp.: There is no effect of x on res1
Alt. hyp.: There is an effect of x on res1

┌──────────────┬────────┬─────────┬─────────────┬───────────┬─────────┬─────────┬─────┐
│ index        ┆ OR     ┆ OR%     ┆ coefficient ┆ std.error ┆ z.value ┆ p.value ┆     │
╞══════════════╪════════╪═════════╪═════════════╪═══════════╪═════════╪═════════╪═════╡
│ Intercept    ┆ 0.03   ┆ -97.0%  ┆ -3.493      ┆ 0.115     ┆ -30.388 ┆ < .001  ┆ *** │
│ zip_bins[1]  ┆ 0.623  ┆ -37.7%  ┆ -0.474      ┆ 0.175     ┆ -2.714  ┆ 0.007   ┆ **  │
│ zip_bins[3]  ┆ 0.642  ┆ -35.8%  ┆ -0.443      ┆ 0.142     ┆ -3.129  ┆ 0.002   ┆ **  │
│ zip_bins[11] ┆ 0.701  ┆ -29.9%  ┆ -0.355      ┆ 0.138     ┆ -2.579  ┆ 0.01    ┆ **  │
│ zip_bins[5]  ┆ 0.635  ┆ -36.5%  ┆ -0.454      ┆ 0.1

Logistic with interactions

In [53]:
clf_int = rsm.model.logistic(
    data={"intuit75k": intuit75k.filter(pl.col("training") == 1)},
    rvar="res1",
    lev="Yes",
    evar=[
        "zip_bins",
        "sex",
        "bizflag",
        "numords",
        "dollars",
        "last",
        "sincepurch",
        "version1",
        "owntaxprod",
        "upgraded",
        "zip801",
        "zip804",
    ],
    ivar=["numords:version1", "last:version1"],
)

clf_int.summary()


Logistic regression (GLM)
Data                 : intuit75k
Response variable    : res1
Level                : Yes
Explanatory variables: zip_bins, sex, bizflag, numords, dollars, last, sincepurch, version1, owntaxprod, upgraded, zip801, zip804
Null hyp.: There is no effect of x on res1
Alt. hyp.: There is an effect of x on res1

┌──────────────────┬────────┬─────────┬─────────────┬───────────┬─────────┬─────────┬─────┐
│ index            ┆ OR     ┆ OR%     ┆ coefficient ┆ std.error ┆ z.value ┆ p.value ┆     │
╞══════════════════╪════════╪═════════╪═════════════╪═══════════╪═════════╪═════════╪═════╡
│ Intercept        ┆ 0.034  ┆ -96.6%  ┆ -3.388      ┆ 0.12      ┆ -28.235 ┆ < .001  ┆ *** │
│ zip_bins[1]      ┆ 0.624  ┆ -37.6%  ┆ -0.472      ┆ 0.175     ┆ -2.695  ┆ 0.007   ┆ **  │
│ zip_bins[3]      ┆ 0.645  ┆ -35.5%  ┆ -0.438      ┆ 0.142     ┆ -3.085  ┆ 0.002   ┆ **  │
│ zip_bins[11]     ┆ 0.7    ┆ -30.0%  ┆ -0.357      ┆ 0.138     ┆ -2.577  ┆ 0.01    ┆ **  │
│ zip_bins[5]      ┆ 0.63


Pseudo R-squared (McFadden): 0.156
Pseudo R-squared (McFadden adjusted): 0.153
Area under the RO Curve (AUC): 0.774
Log-likelihood: -8479.762, AIC: 17027.524, BIC: 17329.055
Chi-squared: 3130.12, df(33), p.value < 0.001
Nr obs: 52,500


In [54]:
intuit75k = intuit75k.with_columns(
    pred_clf=clf.predict(intuit75k).get_column("prediction"),
    pred_clf_int=clf_int.predict(intuit75k).get_column("prediction"),
)
intuit75k

In [55]:
dct = {
    "train": intuit75k.filter(pl.col("training") == 1),
    "test": intuit75k.filter(pl.col("training") == 0),
}
rsm.model.gains_tab(
    dct["train"],
    rvar="res1",
    lev="Yes",
    pred="pred_clf",
)

cum_prop,cum_gains
f64,f64
0.0,0.0
0.1,0.428343
0.2,0.565653
0.3,0.676141
0.4,0.753803
0.5,0.813451
0.6,0.874299
0.7,0.917134
0.8,0.952762


In [56]:
rsm.model.gains_plot(
    dct["train"],
    rvar="res1",
    lev="Yes",
    pred="pred_clf",
)


In [57]:
rsm.model.gains_tab(
    dct["train"],
    rvar="res1",
    lev="Yes",
    pred="pred_clf_int",
)


cum_prop,cum_gains
f64,f64
0.0,0.0
0.1,0.435949
0.2,0.568455
0.3,0.68735
0.4,0.766213
0.5,0.828663
0.6,0.879904
0.7,0.923139
0.8,0.956765


In [58]:
rsm.model.gains_plot(
    dct["train"],
    rvar="res1",
    lev="Yes",
    pred="pred_clf_int",
)


### Evaluating and comparing

In [59]:
MAIL_COST = 1.41
MARGIN = 60.0

p_be = MAIL_COST / MARGIN  # 1.41/60
p_be_50 = MAIL_COST / (0.5 * MARGIN)  # 1.41/30

p_be, p_be_50

(0.0235, 0.047)

In [60]:
test_log1 = intuit75k.filter(pl.col("training") == 0)

test_log1 = test_log1.with_columns(
    [
        # Base break-even decision
        (pl.col("pred_clf") >= p_be).alias("mail_clf"),
        (pl.col("pred_clf_int") >= p_be).alias("mail_clf_int"),
        # 50% rule option A: threshold adjusted
        (pl.col("pred_clf") >= p_be_50).alias("mail_clf_50thr"),
        (pl.col("pred_clf_int") >= p_be_50).alias("mail_clf_int_50thr"),
    ]
)
test_log1.head()


id,zip5,zip_bins,sex,bizflag,numords,dollars,last,sincepurch,version1,owntaxprod,upgraded,res1,training,zip801,zip804,pred_clf,pred_clf_int,mail_clf,mail_clf_int,mail_clf_50thr,mail_clf_int_50thr
i32,str,cat,cat,i32,i32,f64,i32,i32,i32,i32,i32,cat,i32,i8,i8,f64,f64,bool,bool,bool,bool
2,"""53190""","""10""","""Unknown""",0,1,69.5,4,3,0,0,0,"""No""",0,0,0,0.022178,0.023382,false,false,false,false
3,"""37091""","""8""","""Male""",0,4,93.0,14,29,0,0,1,"""No""",0,0,0,0.091385,0.078289,true,true,true,true
5,"""60201""","""11""","""Male""",0,1,24.5,2,3,0,0,0,"""No""",0,0,0,0.025147,0.025852,true,true,false,false
7,"""22980""","""5""","""Male""",0,1,49.5,13,36,1,0,0,"""No""",0,0,0,0.033054,0.021212,true,false,false,false
9,"""34950""","""8""","""Male""",0,1,44.5,15,4,0,0,0,"""No""",0,0,0,0.014706,0.016916,false,false,false,false


In [61]:
cm_mail_clf = (
    test_log1.group_by(["mail_clf", "res1"])
    .agg(pl.len().alias("n"))
    .pivot(values="n", index="mail_clf", on="res1")
    .fill_null(0)
    .with_columns((pl.col("Yes") + pl.col("No")).alias("Total"))
    .select(["mail_clf", "Yes", "No", "Total"])
    .sort("mail_clf", descending=True)
)

cm_mail_clf


mail_clf,Yes,No,Total
bool,u32,u32,u32
true,959,12688,13647
false,144,8709,8853


In [62]:
cm_mail_clf_int = (
    test_log1.group_by(["mail_clf_int", "res1"])
    .agg(pl.len().alias("n"))
    .pivot(values="n", index="mail_clf_int", on="res1")
    .fill_null(0)
    .with_columns((pl.col("Yes") + pl.col("No")).alias("Total"))
    .select(["mail_clf_int", "Yes", "No", "Total"])
    .sort("mail_clf_int", descending=True)
)
cm_mail_clf_int


mail_clf_int,Yes,No,Total
bool,u32,u32,u32
true,946,12168,13114
false,157,9229,9386


In [63]:
cm_mail_clf_50thr = (
    test_log1.group_by(["mail_clf_50thr", "res1"])
    .agg(pl.len().alias("n"))
    .pivot(values="n", index="mail_clf_50thr", on="res1")
    .fill_null(0)
    .with_columns((pl.col("Yes") + pl.col("No")).alias("Total"))
    .select(["mail_clf_50thr", "Yes", "No", "Total"])
    .sort("mail_clf_50thr", descending=True)
)
cm_mail_clf_50thr


mail_clf_50thr,Yes,No,Total
bool,u32,u32,u32
true,743,5737,6480
false,360,15660,16020


In [64]:
cm_mail_clf_int_50thr = (
    test_log1.group_by(["mail_clf_int_50thr", "res1"])
    .agg(pl.len().alias("n"))
    .pivot(values="n", index="mail_clf_int_50thr", on="res1")
    .fill_null(0)
    .with_columns((pl.col("Yes") + pl.col("No")).alias("Total"))
    .select(["mail_clf_int_50thr", "Yes", "No", "Total"])
    .sort("mail_clf_int_50thr", descending=True)
)
cm_mail_clf_int_50thr


mail_clf_int_50thr,Yes,No,Total
bool,u32,u32,u32
true,712,5268,5980
false,391,16129,16520


In [65]:
MAIL_COST = 1.41
MARGIN = 60.0

WAVE1_MAILING = 801_821
ALREADY_PURCHASED_RATE = 0.048
ELIGIBLE_WAVE2 = WAVE1_MAILING * (1 - ALREADY_PURCHASED_RATE)  # 763,334-ish

N_TEST = test_log1.height  # should be 22500

# =========================
# 1) mail_clf (base threshold)
# =========================
cm_mail_clf = (
    test_log1.group_by(["mail_clf", "res1"])
    .agg(pl.len().alias("n"))
    .pivot(values="n", index="mail_clf", on="res1")
    .fill_null(0)
    .with_columns((pl.col("Yes") + pl.col("No")).alias("Total"))
    .select(["mail_clf", "Yes", "No", "Total"])
    .sort("mail_clf", descending=True)
)

mail_true_total = (
    cm_mail_clf.filter(pl.col("mail_clf") == True).select(pl.col("Total")).item()
)
mail_true_yes = (
    cm_mail_clf.filter(pl.col("mail_clf") == True).select(pl.col("Yes")).item()
)

mail_pct = mail_true_total / N_TEST
rate_w1 = mail_true_yes / mail_true_total
rate_w2 = rate_w1 / 2

N_mail = ELIGIBLE_WAVE2 * mail_pct
buyers = N_mail * rate_w2
spend = MAIL_COST * N_mail
profit = MARGIN * buyers - spend
rome = profit / spend

result_mail_clf = pl.DataFrame(
    [
        {
            "rule": "mail_clf",
            "mail_pct": mail_pct,
            "rate_w1_mailed": rate_w1,
            "rate_w2_adj": rate_w2,
            "N_mail_wave2": N_mail,
            "exp_buyers": buyers,
            "spend": spend,
            "profit": profit,
            "ROME": rome,
        }
    ]
)

# =========================
# 2) mail_clf_int (base threshold)
# =========================
cm_mail_clf_int = (
    test_log1.group_by(["mail_clf_int", "res1"])
    .agg(pl.len().alias("n"))
    .pivot(values="n", index="mail_clf_int", on="res1")
    .fill_null(0)
    .with_columns((pl.col("Yes") + pl.col("No")).alias("Total"))
    .select(["mail_clf_int", "Yes", "No", "Total"])
    .sort("mail_clf_int", descending=True)
)

mail_true_total = (
    cm_mail_clf_int.filter(pl.col("mail_clf_int") == True)
    .select(pl.col("Total"))
    .item()
)
mail_true_yes = (
    cm_mail_clf_int.filter(pl.col("mail_clf_int") == True).select(pl.col("Yes")).item()
)

mail_pct = mail_true_total / N_TEST
rate_w1 = mail_true_yes / mail_true_total
rate_w2 = rate_w1 / 2

N_mail = ELIGIBLE_WAVE2 * mail_pct
buyers = N_mail * rate_w2
spend = MAIL_COST * N_mail
profit = MARGIN * buyers - spend
rome = profit / spend

result_mail_clf_int = pl.DataFrame(
    [
        {
            "rule": "mail_clf_int",
            "mail_pct": mail_pct,
            "rate_w1_mailed": rate_w1,
            "rate_w2_adj": rate_w2,
            "N_mail_wave2": N_mail,
            "exp_buyers": buyers,
            "spend": spend,
            "profit": profit,
            "ROME": rome,
        }
    ]
)

# =========================
# 3) mail_clf_50thr (50% threshold)
# =========================
cm_mail_clf_50thr = (
    test_log1.group_by(["mail_clf_50thr", "res1"])
    .agg(pl.len().alias("n"))
    .pivot(values="n", index="mail_clf_50thr", on="res1")
    .fill_null(0)
    .with_columns((pl.col("Yes") + pl.col("No")).alias("Total"))
    .select(["mail_clf_50thr", "Yes", "No", "Total"])
    .sort("mail_clf_50thr", descending=True)
)

mail_true_total = (
    cm_mail_clf_50thr.filter(pl.col("mail_clf_50thr") == True)
    .select(pl.col("Total"))
    .item()
)
mail_true_yes = (
    cm_mail_clf_50thr.filter(pl.col("mail_clf_50thr") == True)
    .select(pl.col("Yes"))
    .item()
)

mail_pct = mail_true_total / N_TEST
rate_w1 = mail_true_yes / mail_true_total
rate_w2 = rate_w1 / 2

N_mail = ELIGIBLE_WAVE2 * mail_pct
buyers = N_mail * rate_w2
spend = MAIL_COST * N_mail
profit = MARGIN * buyers - spend
rome = profit / spend

result_mail_clf_50thr = pl.DataFrame(
    [
        {
            "rule": "mail_clf_50thr",
            "mail_pct": mail_pct,
            "rate_w1_mailed": rate_w1,
            "rate_w2_adj": rate_w2,
            "N_mail_wave2": N_mail,
            "exp_buyers": buyers,
            "spend": spend,
            "profit": profit,
            "ROME": rome,
        }
    ]
)

# =========================
# 4) mail_clf_int_50thr (50% threshold)
# =========================
cm_mail_clf_int_50thr = (
    test_log1.group_by(["mail_clf_int_50thr", "res1"])
    .agg(pl.len().alias("n"))
    .pivot(values="n", index="mail_clf_int_50thr", on="res1")
    .fill_null(0)
    .with_columns((pl.col("Yes") + pl.col("No")).alias("Total"))
    .select(["mail_clf_int_50thr", "Yes", "No", "Total"])
    .sort("mail_clf_int_50thr", descending=True)
)

mail_true_total = (
    cm_mail_clf_int_50thr.filter(pl.col("mail_clf_int_50thr") == True)
    .select(pl.col("Total"))
    .item()
)
mail_true_yes = (
    cm_mail_clf_int_50thr.filter(pl.col("mail_clf_int_50thr") == True)
    .select(pl.col("Yes"))
    .item()
)

mail_pct = mail_true_total / N_TEST
rate_w1 = mail_true_yes / mail_true_total
rate_w2 = rate_w1 / 2

N_mail = ELIGIBLE_WAVE2 * mail_pct
buyers = N_mail * rate_w2
spend = MAIL_COST * N_mail
profit = MARGIN * buyers - spend
rome = profit / spend

result_mail_clf_int_50thr = pl.DataFrame(
    [
        {
            "rule": "mail_clf_int_50thr",
            "mail_pct": mail_pct,
            "rate_w1_mailed": rate_w1,
            "rate_w2_adj": rate_w2,
            "N_mail_wave2": N_mail,
            "exp_buyers": buyers,
            "spend": spend,
            "profit": profit,
            "ROME": rome,
        }
    ]
)

# =========================
# Combine and rank
# =========================
results = pl.concat(
    [
        result_mail_clf,
        result_mail_clf_int,
        result_mail_clf_50thr,
        result_mail_clf_int_50thr,
    ]
)

results.sort("profit", descending=True)


rule,mail_pct,rate_w1_mailed,rate_w2_adj,N_mail_wave2,exp_buyers,spend,profit,ROME
str,f64,f64,f64,f64,f64,f64,f64,f64
"""mail_clf_50thr""",0.288,0.11466,0.05733,219840.074496,12603.485752,309974.505039,446234.640102,1.439585
"""mail_clf_int_50thr""",0.265778,0.119064,0.059532,202877.105785,12077.633722,286056.719157,438601.304182,1.533267
"""mail_clf_int""",0.582844,0.072137,0.036068,444904.743355,16046.968401,627315.688131,335502.415912,0.534822
"""mail_clf""",0.606533,0.070272,0.035136,462987.268001,16267.486994,652812.047882,323237.171756,0.495146


## Logistic Rule Evaluation

The logistic-family mailing rules perform well because they capture stable upgrade patterns and convert them into value-aware mailing decisions. The most important takeaway is that tightening the mailing rule can improve economics materially: broader mailings increase buyers, but selective mailings improve **profit quality** by reducing waste.


### Neural Network (MLP)

In [66]:
clf_mlp = rsm.model.mlp(
    data={"intuit75k": train.filter(pl.col("training") == 1)},
    rvar="res1",
    lev="Yes",
    evar=[
        "zip_bins",
        "sex",
        "bizflag",
        "numords",
        "dollars",
        "last",
        "sincepurch",
        "version1",
        "owntaxprod",
        "upgraded",
        "zip801",
        "zip804",
    ],
    hidden_layer_sizes=(1,),
    alpha=0.001,
    max_iter=100_000,
    random_state=1234,
)

clf_mlp.summary()


Multi-layer Perceptron (NN)
Data                 : intuit75k
Response variable    : res1
Level                : Yes
Explanatory variables: zip_bins, sex, bizflag, numords, dollars, last, sincepurch, version1, owntaxprod, upgraded, zip801, zip804
Model type           : classification
Nr. of features      : (12, 31)
Nr. of weights       : 34
Nr. of observations  : 52,500
Hidden_layer_sizes   : (1,)
Activation function  : tanh
Solver               : lbfgs
Alpha                : 0.001
Batch size           : auto
Learning rate        : 0.001
Maximum iterations   : 100000
random_state         : 1234
AUC                  : 0.768

Raw data             :
shape: (5, 12)
┌──────────┬──────┬─────────┬─────────┬─────────┬──────┬────────────┬──────────┬────────────┬──────────┬────────┬────────┐
│ zip_bins ┆ sex  ┆ bizflag ┆ numords ┆ dollars ┆ last ┆ sincepurch ┆ version1 ┆ owntaxprod ┆ upgraded ┆ zip801 ┆ zip804 │
│ ---      ┆ ---  ┆ ---     ┆ ---     ┆ ---     ┆ ---  ┆ ---        ┆ ---      ┆ --- 

In [67]:
hls = [(i,) for i in range(1, 11)]
param_grid = {
    "hidden_layer_sizes": hls,
    "alpha": pl.linear_space(0, 1, 11, eager=True).round(1).to_list(),
}
scoring = {"AUC": "roc_auc"}
param_grid

{'hidden_layer_sizes': [(1,),
  (2,),
  (3,),
  (4,),
  (5,),
  (6,),
  (7,),
  (8,),
  (9,),
  (10,)],
 'alpha': [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]}

In [68]:
stratified_k_fold = StratifiedKFold(n_splits=5, shuffle=True, random_state=1234)
cv = GridSearchCV(
    clf_mlp.fitted,
    param_grid,
    scoring=scoring,
    cv=stratified_k_fold,
    n_jobs=1,
    refit=list(scoring.keys())[0],
    verbose=0,
).fit(clf_mlp.data_onehot, clf_mlp.data.get_column("res1"))

In [69]:
cv.best_params_

{'alpha': 0.0, 'hidden_layer_sizes': (2,)}

In [70]:
cv.best_score_

np.float64(0.7716072535551054)

In [71]:
clf_mlpcv = rsm.model.mlp(
    data={"intuit75k": train.filter(pl.col("training") == 1)},
    rvar="res1",
    lev="Yes",
    evar=[
        "zip_bins",
        "sex",
        "bizflag",
        "numords",
        "dollars",
        "last",
        "sincepurch",
        "version1",
        "owntaxprod",
        "upgraded",
        "zip801",
        "zip804",
    ],
    random_state=1234,
    **cv.best_params_,
)

clf_mlpcv.summary()


Multi-layer Perceptron (NN)
Data                 : intuit75k
Response variable    : res1
Level                : Yes
Explanatory variables: zip_bins, sex, bizflag, numords, dollars, last, sincepurch, version1, owntaxprod, upgraded, zip801, zip804
Model type           : classification
Nr. of features      : (12, 31)
Nr. of weights       : 67
Nr. of observations  : 52,500
Hidden_layer_sizes   : (2,)
Activation function  : tanh
Solver               : lbfgs
Alpha                : 0.0
Batch size           : auto
Learning rate        : 0.001
Maximum iterations   : 1000000
random_state         : 1234
AUC                  : 0.777

Raw data             :
shape: (5, 12)
┌──────────┬──────┬─────────┬─────────┬─────────┬──────┬────────────┬──────────┬────────────┬──────────┬────────┬────────┐
│ zip_bins ┆ sex  ┆ bizflag ┆ numords ┆ dollars ┆ last ┆ sincepurch ┆ version1 ┆ owntaxprod ┆ upgraded ┆ zip801 ┆ zip804 │
│ ---      ┆ ---  ┆ ---     ┆ ---     ┆ ---     ┆ ---  ┆ ---        ┆ ---      ┆ ---  

In [72]:
intuit75k = intuit75k.with_columns(
    pred_mlpcv=clf_mlpcv.predict(intuit75k).get_column("prediction")
)
intuit75k.head()


id,zip5,zip_bins,sex,bizflag,numords,dollars,last,sincepurch,version1,owntaxprod,upgraded,res1,training,zip801,zip804,pred_clf,pred_clf_int,pred_mlpcv
i32,str,cat,cat,i32,i32,f64,i32,i32,i32,i32,i32,cat,i32,i8,i8,f64,f64,f64
1,"""94553""","""18""","""Male""",0,2,109.5,5,12,0,0,0,"""No""",1,0,0,0.044161,0.042763,0.043869
2,"""53190""","""10""","""Unknown""",0,1,69.5,4,3,0,0,0,"""No""",0,0,0,0.022178,0.023382,0.031703
3,"""37091""","""8""","""Male""",0,4,93.0,14,29,0,0,1,"""No""",0,0,0,0.091385,0.078289,0.072092
4,"""02125""","""1""","""Male""",0,1,22.0,17,1,0,0,0,"""No""",1,0,0,0.011381,0.013703,0.011184
5,"""60201""","""11""","""Male""",0,1,24.5,2,3,0,0,0,"""No""",0,0,0,0.025147,0.025852,0.021894


In [73]:
test_log1 = intuit75k.filter(pl.col("training") == 0)

test_log1 = test_log1.with_columns(
    [
        (pl.col("pred_mlpcv") >= p_be).alias("mail_mlpcv"),
        (pl.col("pred_mlpcv") >= p_be_50).alias("mail_mlpcv_50thr"),
    ]
)


In [74]:
cm_mail_mlpcv = (
    test_log1.group_by(["mail_mlpcv", "res1"])
    .agg(pl.len().alias("n"))
    .pivot(values="n", index="mail_mlpcv", on="res1")
    .fill_null(0)
    .with_columns((pl.col("Yes") + pl.col("No")).alias("Total"))
    .select(["mail_mlpcv", "Yes", "No", "Total"])
    .sort("mail_mlpcv", descending=True)
)
cm_mail_mlpcv


mail_mlpcv,Yes,No,Total
bool,u32,u32,u32
true,938,12157,13095
false,165,9240,9405


In [75]:
cm_mail_mlpcv_50thr = (
    test_log1.group_by(["mail_mlpcv_50thr", "res1"])
    .agg(pl.len().alias("n"))
    .pivot(values="n", index="mail_mlpcv_50thr", on="res1")
    .fill_null(0)
    .with_columns((pl.col("Yes") + pl.col("No")).alias("Total"))
    .select(["mail_mlpcv_50thr", "Yes", "No", "Total"])
    .sort("mail_mlpcv_50thr", descending=True)
)
cm_mail_mlpcv_50thr


mail_mlpcv_50thr,Yes,No,Total
bool,u32,u32,u32
true,741,5734,6475
false,362,15663,16025


In [76]:
mail_true_total = (
    cm_mail_mlpcv.filter(pl.col("mail_mlpcv") == True).select("Total").item()
)
mail_true_yes = cm_mail_mlpcv.filter(pl.col("mail_mlpcv") == True).select("Yes").item()

mail_pct = mail_true_total / N_TEST
rate_w1 = mail_true_yes / mail_true_total
rate_w2 = rate_w1 / 2

N_mail = ELIGIBLE_WAVE2 * mail_pct
buyers = N_mail * rate_w2
spend = MAIL_COST * N_mail
profit = MARGIN * buyers - spend
rome = profit / spend

result_mail_mlpcv = pl.DataFrame(
    [
        {
            "rule": "mail_mlpcv",
            "mail_pct": mail_pct,
            "rate_w1_mailed": rate_w1,
            "rate_w2_adj": rate_w2,
            "N_mail_wave2": N_mail,
            "exp_buyers": buyers,
            "spend": spend,
            "profit": profit,
            "ROME": rome,
        }
    ]
)

result_mail_mlpcv


rule,mail_pct,rate_w1_mailed,rate_w2_adj,N_mail_wave2,exp_buyers,spend,profit,ROME
str,f64,f64,f64,f64,f64,f64,f64,f64
"""mail_mlpcv""",0.582,0.07163,0.035815,444260.150544,15911.264651,626406.812267,328269.066794,0.524051


In [77]:
mail_true_total = (
    cm_mail_mlpcv_50thr.filter(pl.col("mail_mlpcv_50thr") == True)
    .select("Total")
    .item()
)
mail_true_yes = (
    cm_mail_mlpcv_50thr.filter(pl.col("mail_mlpcv_50thr") == True).select("Yes").item()
)

mail_pct = mail_true_total / N_TEST
rate_w1 = mail_true_yes / mail_true_total
rate_w2 = rate_w1 / 2

N_mail = ELIGIBLE_WAVE2 * mail_pct
buyers = N_mail * rate_w2
spend = MAIL_COST * N_mail
profit = MARGIN * buyers - spend
rome = profit / spend

result_mail_mlpcv_50thr = pl.DataFrame(
    [
        {
            "rule": "mail_mlpcv_50thr",
            "mail_pct": mail_pct,
            "rate_w1_mailed": rate_w1,
            "rate_w2_adj": rate_w2,
            "N_mail_wave2": N_mail,
            "exp_buyers": buyers,
            "spend": spend,
            "profit": profit,
            "ROME": rome,
        }
    ]
)

result_mail_mlpcv_50thr


rule,mail_pct,rate_w1_mailed,rate_w2_adj,N_mail_wave2,exp_buyers,spend,profit,ROME
str,f64,f64,f64,f64,f64,f64,f64,f64
"""mail_mlpcv_50thr""",0.287778,0.11444,0.05722,219670.444809,12569.559815,309735.327181,444438.261715,1.434897


In [78]:
pl.concat([result_mail_mlpcv, result_mail_mlpcv_50thr]).sort("profit", descending=True)


rule,mail_pct,rate_w1_mailed,rate_w2_adj,N_mail_wave2,exp_buyers,spend,profit,ROME
str,f64,f64,f64,f64,f64,f64,f64,f64
"""mail_mlpcv_50thr""",0.287778,0.11444,0.05722,219670.444809,12569.559815,309735.327181,444438.261715,1.434897
"""mail_mlpcv""",0.582,0.07163,0.035815,444260.150544,15911.264651,626406.812267,328269.066794,0.524051


## MLP Interpretation

The tuned neural network improves flexibility and remains competitive on profit, especially under the tighter half-threshold rule. That suggests there is real nonlinear structure in the response problem, but the business case for using the model still depends on deployment economics, not just fit statistics. A model that looks sophisticated but does not materially improve profit should not be preferred simply because it is more complex.


### Random Forests

In [79]:
clf_rf = rsm.model.rforest(
    data={"intuit75k": train.filter(pl.col("training") == 1)},
    rvar="res1",
    lev="Yes",
    evar=[
        "zip_bins",
        "sex",
        "bizflag",
        "numords",
        "dollars",
        "last",
        "sincepurch",
        "version1",
        "owntaxprod",
        "upgraded",
        "zip801",
        "zip804",
    ],
    max_features=2,
    n_estimators=100,
)

clf_rf.summary()


Random Forest
Data                 : intuit75k
Response variable    : res1
Level                : Yes
Explanatory variables: zip_bins, sex, bizflag, numords, dollars, last, sincepurch, version1, owntaxprod, upgraded, zip801, zip804
OOB                  : True
Model type           : classification
Nr. of features      : (12, 33)
Nr. of observations  : 52,500
max_features         : 2 (2)
n_estimators         : 100
min_samples_leaf     : 1
max_samples          : 1.0
random_state         : 1234
AUC                  : 0.692

Estimation data      :
shape: (5, 33)
┌─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┐
│ biz ┆ num ┆ dol ┆ las ┆ sin ┆ ver ┆ own ┆ upg ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ sex ┆ sex ┆ sex │
│ fla ┆ ord ┆ lar ┆ t   ┆ cep ┆ sio 

In [80]:
param_grid = {
    "max_features": list(range(1, 6)),
    "n_estimators": pl.arange(100, 600, 100, eager=True).to_list(),
}
scoring = {"AUC": "roc_auc"}
param_grid


{'max_features': [1, 2, 3, 4, 5], 'n_estimators': [100, 200, 300, 400, 500]}

In [81]:
stratified_k_fold = StratifiedKFold(n_splits=5, shuffle=True, random_state=1234)
cv = GridSearchCV(
    clf_rf.fitted,
    param_grid,
    scoring=scoring,
    cv=stratified_k_fold,
    n_jobs=1,
    refit=list(scoring.keys())[0],
    verbose=0,
).fit(clf_rf.data_onehot, clf_rf.data.get_column("res1"))


In [82]:
cv.best_params_

{'max_features': 5, 'n_estimators': 500}

In [83]:
cv.best_score_

np.float64(0.7293044974936173)

In [84]:
clf_rfcv = rsm.model.rforest(
    data={"intuit75k": train.filter(pl.col("training") == 1)},
    rvar="res1",
    lev="Yes",
    evar=[
        "zip_bins",
        "sex",
        "bizflag",
        "numords",
        "dollars",
        "last",
        "sincepurch",
        "version1",
        "owntaxprod",
        "upgraded",
        "zip801",
        "zip804",
    ],
    random_state=1234,
    **cv.best_params_,
)

clf_rfcv.summary()


Random Forest
Data                 : intuit75k
Response variable    : res1
Level                : Yes
Explanatory variables: zip_bins, sex, bizflag, numords, dollars, last, sincepurch, version1, owntaxprod, upgraded, zip801, zip804
OOB                  : True
Model type           : classification
Nr. of features      : (12, 33)
Nr. of observations  : 52,500
max_features         : 5 (5)
n_estimators         : 500
min_samples_leaf     : 1
max_samples          : 1.0
random_state         : 1234
AUC                  : 0.728

Estimation data      :
shape: (5, 33)
┌─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┐
│ biz ┆ num ┆ dol ┆ las ┆ sin ┆ ver ┆ own ┆ upg ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ sex ┆ sex ┆ sex │
│ fla ┆ ord ┆ lar ┆ t   ┆ cep ┆ sio 

In [85]:
intuit75k = intuit75k.with_columns(
    pred_rf=clf_rfcv.predict(intuit75k).get_column("prediction")
)
intuit75k.head()

id,zip5,zip_bins,sex,bizflag,numords,dollars,last,sincepurch,version1,owntaxprod,upgraded,res1,training,zip801,zip804,pred_clf,pred_clf_int,pred_mlpcv,pred_rf
i32,str,cat,cat,i32,i32,f64,i32,i32,i32,i32,i32,cat,i32,i8,i8,f64,f64,f64,f64
1,"""94553""","""18""","""Male""",0,2,109.5,5,12,0,0,0,"""No""",1,0,0,0.044161,0.042763,0.043869,0.022
2,"""53190""","""10""","""Unknown""",0,1,69.5,4,3,0,0,0,"""No""",0,0,0,0.022178,0.023382,0.031703,0.026
3,"""37091""","""8""","""Male""",0,4,93.0,14,29,0,0,1,"""No""",0,0,0,0.091385,0.078289,0.072092,0.102
4,"""02125""","""1""","""Male""",0,1,22.0,17,1,0,0,0,"""No""",1,0,0,0.011381,0.013703,0.011184,0.009
5,"""60201""","""11""","""Male""",0,1,24.5,2,3,0,0,0,"""No""",0,0,0,0.025147,0.025852,0.021894,0.028


In [86]:
test_log1 = intuit75k.filter(pl.col("training") == 0)

test_log1 = test_log1.with_columns(
    (pl.col("pred_rf") >= p_be).alias("mail_rf"),
    (pl.col("pred_rf") >= p_be_50).alias("mail_rf_50thr"),
)


In [87]:
cm_mail_rf = (
    test_log1.group_by(["mail_rf", "res1"])
    .agg(pl.len().alias("n"))
    .pivot(values="n", index="mail_rf", on="res1")
    .fill_null(0)
    .with_columns((pl.col("Yes") + pl.col("No")).alias("Total"))
    .select(["mail_rf", "Yes", "No", "Total"])
    .sort("mail_rf", descending=True)
)
cm_mail_rf


mail_rf,Yes,No,Total
bool,u32,u32,u32
true,821,9575,10396
false,282,11822,12104


In [88]:
cm_mail_rf_50thr = (
    test_log1.group_by(["mail_rf_50thr", "res1"])
    .agg(pl.len().alias("n"))
    .pivot(values="n", index="mail_rf_50thr", on="res1")
    .fill_null(0)
    .with_columns((pl.col("Yes") + pl.col("No")).alias("Total"))
    .select(["mail_rf_50thr", "Yes", "No", "Total"])
    .sort("mail_rf_50thr", descending=True)
)
cm_mail_rf_50thr


mail_rf_50thr,Yes,No,Total
bool,u32,u32,u32
true,665,5794,6459
false,438,15603,16041


In [89]:
# -----------------------
# Random Forest (p_be) rule: mail_rf
# -----------------------
mail_true_total = cm_mail_rf.filter(pl.col("mail_rf") == True).select("Total").item()
mail_true_yes = cm_mail_rf.filter(pl.col("mail_rf") == True).select("Yes").item()

mail_pct = mail_true_total / N_TEST
rate_w1 = mail_true_yes / mail_true_total
rate_w2 = rate_w1 / 2

N_mail = ELIGIBLE_WAVE2 * mail_pct
buyers = N_mail * rate_w2
spend = MAIL_COST * N_mail
profit = MARGIN * buyers - spend
rome = profit / spend

result_mail_rf = pl.DataFrame(
    [
        {
            "rule": "mail_rf",
            "mail_pct": mail_pct,
            "rate_w1_mailed": rate_w1,
            "rate_w2_adj": rate_w2,
            "N_mail_wave2": N_mail,
            "exp_buyers": buyers,
            "spend": spend,
            "profit": profit,
            "ROME": rome,
        }
    ]
)

result_mail_rf


rule,mail_pct,rate_w1_mailed,rate_w2_adj,N_mail_wave2,exp_buyers,spend,profit,ROME
str,f64,f64,f64,f64,f64,f64,f64,f64
"""mail_rf""",0.462044,0.078973,0.039486,352694.045441,13926.597312,497298.604072,338297.234637,0.68027


In [90]:
# -----------------------
# Random Forest (0.5*p_be) rule: mail_rf_50thr
# -----------------------
mail_true_total = (
    cm_mail_rf_50thr.filter(pl.col("mail_rf_50thr") == True).select("Total").item()
)
mail_true_yes = (
    cm_mail_rf_50thr.filter(pl.col("mail_rf_50thr") == True).select("Yes").item()
)

mail_pct = mail_true_total / N_TEST
rate_w1 = mail_true_yes / mail_true_total
rate_w2 = rate_w1 / 2

N_mail = ELIGIBLE_WAVE2 * mail_pct
buyers = N_mail * rate_w2
spend = MAIL_COST * N_mail
profit = MARGIN * buyers - spend
rome = profit / spend

result_mail_rf_50thr = pl.DataFrame(
    [
        {
            "rule": "mail_rf_50thr",
            "mail_pct": mail_pct,
            "rate_w1_mailed": rate_w1,
            "rate_w2_adj": rate_w2,
            "N_mail_wave2": N_mail,
            "exp_buyers": buyers,
            "spend": spend,
            "profit": profit,
            "ROME": rome,
        }
    ]
)

result_mail_rf_50thr


rule,mail_pct,rate_w1_mailed,rate_w2_adj,N_mail_wave2,exp_buyers,spend,profit,ROME
str,f64,f64,f64,f64,f64,f64,f64,f64
"""mail_rf_50thr""",0.287067,0.102957,0.051479,219127.62981,11280.374193,308969.958032,367852.493541,1.190577


In [91]:
pl.concat([result_mail_rf, result_mail_rf_50thr]).sort("profit", descending=True)


rule,mail_pct,rate_w1_mailed,rate_w2_adj,N_mail_wave2,exp_buyers,spend,profit,ROME
str,f64,f64,f64,f64,f64,f64,f64,f64
"""mail_rf_50thr""",0.287067,0.102957,0.051479,219127.62981,11280.374193,308969.958032,367852.493541,1.190577
"""mail_rf""",0.462044,0.078973,0.039486,352694.045441,13926.597312,497298.604072,338297.234637,0.68027


## Random Forest Interpretation

Random Forest provides a strong tree-based benchmark but underperforms the best alternatives on final campaign profit. In business language, this means the model can find responders, but not as efficiently as the top policies once mailing cost is considered. It is therefore informative as a comparison model, but not the recommended operational choice.


### XGBoost Model

In [92]:
clf_xgb = rsm.model.xgboost(
    data={"intuit75k": train.filter(pl.col("training") == 1)},
    rvar="res1",
    lev="Yes",
    evar=[
        "zip_bins",
        "sex",
        "bizflag",
        "numords",
        "dollars",
        "last",
        "sincepurch",
        "version1",
        "owntaxprod",
        "upgraded",
        "zip801",
        "zip804",
    ],
    n_estimators=100,
    max_depth=6,
    min_child_weight=1,
    learning_rate=0.3,
    random_state=1234,
)

clf_xgb.summary()


XGBoost
Data                 : intuit75k
Response variable    : res1
Level                : Yes
Explanatory variables: zip_bins, sex, bizflag, numords, dollars, last, sincepurch, version1, owntaxprod, upgraded, zip801, zip804
Model type           : classification
Nr. of features      : (12, 33)
Nr. of observations  : 52,500
n_estimators         : 100
max_depth            : 6
min_child_weight     : 1
learning_rate        : 0.3
subsample            : 1.0
colsample_bytree     : 1.0
random_state         : 1234
AUC                  : 0.927

Estimation data      :
shape: (5, 33)
┌─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┐
│ biz ┆ num ┆ dol ┆ las ┆ sin ┆ ver ┆ own ┆ upg ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ sex ┆ sex ┆ sex │
│ fla ┆ ord ┆ lar ┆ 

In [93]:
param_grid = {
    "n_estimators": [50, 100, 150],
    "max_depth": list(range(7)),
    "min_child_weight": [1, 5, 10],
    "learning_rate": [0.1, 0.2, 0.3],
}
scoring = {"AUC": "roc_auc"}
param_grid


{'n_estimators': [50, 100, 150],
 'max_depth': [0, 1, 2, 3, 4, 5, 6],
 'min_child_weight': [1, 5, 10],
 'learning_rate': [0.1, 0.2, 0.3]}

In [94]:
stratified_k_fold = StratifiedKFold(n_splits=5, shuffle=True, random_state=1234)
cv = GridSearchCV(
    clf_xgb.fitted,
    param_grid,
    scoring=scoring,
    cv=stratified_k_fold,
    n_jobs=1,
    refit=list(scoring.keys())[0],
    verbose=0,
).fit(clf_xgb.data_onehot, clf_xgb.data.get_column("res1"))


In [95]:
cv.best_params_

{'learning_rate': 0.1,
 'max_depth': 3,
 'min_child_weight': 5,
 'n_estimators': 100}

In [96]:
cv.best_score_

np.float64(0.77087754443536)

In [97]:
clf_xgbcv = rsm.model.xgboost(
    data={"intuit75k": train.filter(pl.col("training") == 1)},
    rvar="res1",
    lev="Yes",
    evar=[
        "zip_bins",
        "sex",
        "bizflag",
        "numords",
        "dollars",
        "last",
        "sincepurch",
        "version1",
        "owntaxprod",
        "upgraded",
        "zip801",
        "zip804",
    ],
    random_state=1234,
    **cv.best_params_,
)

clf_xgbcv.summary()


XGBoost
Data                 : intuit75k
Response variable    : res1
Level                : Yes
Explanatory variables: zip_bins, sex, bizflag, numords, dollars, last, sincepurch, version1, owntaxprod, upgraded, zip801, zip804
Model type           : classification
Nr. of features      : (12, 33)
Nr. of observations  : 52,500
n_estimators         : 100
max_depth            : 3
min_child_weight     : 5
learning_rate        : 0.1
subsample            : 1.0
colsample_bytree     : 1.0
random_state         : 1234
AUC                  : 0.788

Estimation data      :
shape: (5, 33)
┌─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┐
│ biz ┆ num ┆ dol ┆ las ┆ sin ┆ ver ┆ own ┆ upg ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ zip ┆ sex ┆ sex ┆ sex │
│ fla ┆ ord ┆ lar ┆ 

In [98]:
intuit75k = intuit75k.with_columns(
    pred_xgb=clf_xgbcv.predict(intuit75k).get_column("prediction")
)
intuit75k.head()

id,zip5,zip_bins,sex,bizflag,numords,dollars,last,sincepurch,version1,owntaxprod,upgraded,res1,training,zip801,zip804,pred_clf,pred_clf_int,pred_mlpcv,pred_rf,pred_xgb
i32,str,cat,cat,i32,i32,f64,i32,i32,i32,i32,i32,cat,i32,i8,i8,f64,f64,f64,f64,f32
1,"""94553""","""18""","""Male""",0,2,109.5,5,12,0,0,0,"""No""",1,0,0,0.044161,0.042763,0.043869,0.022,0.032531
2,"""53190""","""10""","""Unknown""",0,1,69.5,4,3,0,0,0,"""No""",0,0,0,0.022178,0.023382,0.031703,0.026,0.029673
3,"""37091""","""8""","""Male""",0,4,93.0,14,29,0,0,1,"""No""",0,0,0,0.091385,0.078289,0.072092,0.102,0.084245
4,"""02125""","""1""","""Male""",0,1,22.0,17,1,0,0,0,"""No""",1,0,0,0.011381,0.013703,0.011184,0.009,0.019164
5,"""60201""","""11""","""Male""",0,1,24.5,2,3,0,0,0,"""No""",0,0,0,0.025147,0.025852,0.021894,0.028,0.025577


In [99]:
test_xgb = intuit75k.filter(pl.col("training") == 0)

test_xgb = test_xgb.with_columns(
    [
        (pl.col("pred_xgb") >= p_be).alias("mail_xgb"),
        (pl.col("pred_xgb") >= p_be_50).alias("mail_xgb_50thr"),
    ]
)


In [100]:
cm_mail_xgb = (
    test_xgb.group_by(["mail_xgb", "res1"])
    .agg(pl.len().alias("n"))
    .pivot(values="n", index="mail_xgb", on="res1")
    .fill_null(0)
    .with_columns((pl.col("Yes") + pl.col("No")).alias("Total"))
    .select(["mail_xgb", "Yes", "No", "Total"])
    .sort("mail_xgb", descending=True)
)
cm_mail_xgb


mail_xgb,Yes,No,Total
bool,u32,u32,u32
true,941,11781,12722
false,162,9616,9778


In [101]:
cm_mail_xgb_50thr = (
    test_xgb.group_by(["mail_xgb_50thr", "res1"])
    .agg(pl.len().alias("n"))
    .pivot(values="n", index="mail_xgb_50thr", on="res1")
    .fill_null(0)
    .with_columns((pl.col("Yes") + pl.col("No")).alias("Total"))
    .select(["mail_xgb_50thr", "Yes", "No", "Total"])
    .sort("mail_xgb_50thr", descending=True)
)
cm_mail_xgb_50thr


mail_xgb_50thr,Yes,No,Total
bool,u32,u32,u32
true,736,5527,6263
false,367,15870,16237


In [102]:
mail_true_total = cm_mail_xgb.filter(pl.col("mail_xgb") == True).select("Total").item()
mail_true_yes = cm_mail_xgb.filter(pl.col("mail_xgb") == True).select("Yes").item()

mail_pct = mail_true_total / N_TEST
rate_w1 = mail_true_yes / mail_true_total
rate_w2 = rate_w1 / 2

N_mail = ELIGIBLE_WAVE2 * mail_pct
buyers = N_mail * rate_w2
spend = MAIL_COST * N_mail
profit = MARGIN * buyers - spend
rome = profit / spend

result_mail_xgb = pl.DataFrame(
    [
        {
            "rule": "mail_xgb",
            "mail_pct": mail_pct,
            "rate_w1_mailed": rate_w1,
            "rate_w2_adj": rate_w2,
            "N_mail_wave2": N_mail,
            "exp_buyers": buyers,
            "spend": spend,
            "profit": profit,
            "ROME": rome,
        }
    ]
)
result_mail_xgb

# Profit + ROME for mail_xgb_50thr
mail_true_total = (
    cm_mail_xgb_50thr.filter(pl.col("mail_xgb_50thr") == True).select("Total").item()
)
mail_true_yes = (
    cm_mail_xgb_50thr.filter(pl.col("mail_xgb_50thr") == True).select("Yes").item()
)

mail_pct = mail_true_total / N_TEST
rate_w1 = mail_true_yes / mail_true_total
rate_w2 = rate_w1 / 2

N_mail = ELIGIBLE_WAVE2 * mail_pct
buyers = N_mail * rate_w2
spend = MAIL_COST * N_mail
profit = MARGIN * buyers - spend
rome = profit / spend

result_mail_xgb_50thr = pl.DataFrame(
    [
        {
            "rule": "mail_xgb_50thr",
            "mail_pct": mail_pct,
            "rate_w1_mailed": rate_w1,
            "rate_w2_adj": rate_w2,
            "N_mail_wave2": N_mail,
            "exp_buyers": buyers,
            "spend": spend,
            "profit": profit,
            "ROME": rome,
        }
    ]
)
result_mail_xgb_50thr

# Comparing
pl.concat([result_mail_xgb, result_mail_xgb_50thr]).sort("profit", descending=True)


rule,mail_pct,rate_w1_mailed,rate_w2_adj,N_mail_wave2,exp_buyers,spend,profit,ROME
str,f64,f64,f64,f64,f64,f64,f64,f64
"""mail_xgb_50thr""",0.278356,0.117516,0.058758,212478.146075,12484.744971,299594.185966,449490.512316,1.500331
"""mail_xgb""",0.565422,0.073966,0.036983,431605.775886,15962.153557,608564.143999,349165.069431,0.573752


## XGBoost Interpretation

XGBoost delivers the strongest overall deployment result because it balances ranking power with selective targeting. The half-threshold rule is especially important: it captures enough high-probability responders to maximize profit while avoiding the heavy spend associated with mailing most of the file. This is the clearest business recommendation in the notebook because it translates directly into a wave-2 action plan.


Comparing results across all models

In [103]:
results_all = pl.concat(
    [
        # logistic (you already created these)
        result_mail_clf,
        result_mail_clf_50thr,
        result_mail_clf_int,
        result_mail_clf_int_50thr,
        # mlp
        result_mail_mlpcv,
        result_mail_mlpcv_50thr,
        # random forest
        result_mail_rf,
        result_mail_rf_50thr,
        # xgboost
        result_mail_xgb,
        result_mail_xgb_50thr,
    ]
).sort("profit", descending=True)

results_all


rule,mail_pct,rate_w1_mailed,rate_w2_adj,N_mail_wave2,exp_buyers,spend,profit,ROME
str,f64,f64,f64,f64,f64,f64,f64,f64
"""mail_xgb_50thr""",0.278356,0.117516,0.058758,212478.146075,12484.744971,299594.185966,449490.512316,1.500331
"""mail_clf_50thr""",0.288,0.11466,0.05733,219840.074496,12603.485752,309974.505039,446234.640102,1.439585
"""mail_mlpcv_50thr""",0.287778,0.11444,0.05722,219670.444809,12569.559815,309735.327181,444438.261715,1.434897
"""mail_clf_int_50thr""",0.265778,0.119064,0.059532,202877.105785,12077.633722,286056.719157,438601.304182,1.533267
"""mail_rf_50thr""",0.287067,0.102957,0.051479,219127.62981,11280.374193,308969.958032,367852.493541,1.190577
"""mail_xgb""",0.565422,0.073966,0.036983,431605.775886,15962.153557,608564.143999,349165.069431,0.573752
"""mail_rf""",0.462044,0.078973,0.039486,352694.045441,13926.597312,497298.604072,338297.234637,0.68027
"""mail_clf_int""",0.582844,0.072137,0.036068,444904.743355,16046.968401,627315.688131,335502.415912,0.534822
"""mail_mlpcv""",0.582,0.07163,0.035815,444260.150544,15911.264651,626406.812267,328269.066794,0.524051


## Cross-Model Business Comparison

Looking across all models, the main lesson is that model choice and mailing rule must be evaluated together. A model with a slightly stronger AUC is only strategically better if it converts into higher campaign profit or higher ROME. Here, the tuned XGBoost half-threshold rule stands out because it combines high expected profit with a disciplined mailing share, making it more attractive operationally than broader, lower-efficiency rules.


Writing CSV

In [104]:
# Add the decision columns from the XGB predicted probabilities
intuit75k = intuit75k.with_columns(
    [
        (pl.col("pred_xgb") >= p_be).alias("mail_xgb"),
        (pl.col("pred_xgb") >= p_be_50).alias("mail_xgb_50thr"),
    ]
)

intuit75k.select(["pred_xgb", "mail_xgb", "mail_xgb_50thr"]).head()


pred_xgb,mail_xgb,mail_xgb_50thr
f32,bool,bool
0.032531,true,false
0.029673,true,false
0.084245,true,true
0.019164,false,false
0.025577,true,false


In [105]:
submit_df = intuit75k.filter(pl.col("training") == 0).select(
    [
        pl.col("id"),
        pl.col("mail_xgb_50thr").cast(pl.Boolean).alias("mailto_wave2"),
    ]
)

print("rows:", submit_df.height)  # should be 22500
submit_df.get_column("mailto_wave2").value_counts()


rows: 22500


mailto_wave2,count
bool,u32
false,16237
true,6263


## Deployment Output

The final CSV step shows how the analysis becomes executable marketing output: each customer in the wave-2 eligible set is converted into a binary mailing decision. This is the bridge from analytics to campaign execution, and it demonstrates that the project is not just a modeling exercise but a deployable targeting workflow.


In [106]:
submit_df.write_csv("mailto_wave2.csv")
